# DAPTA · Qwen3-VL-8B CORD QLoRA

Kernel: **Select Kernel → Colab → New Colab Server → A100**. Run cell 1, then **save (⌘S)** so Claude can read the output.


In [2]:
# Cell 1: where is this running?
import platform, socket, os
print('host:', socket.gethostname(), '|', platform.platform())
print('colab:', any(k.startswith('COLAB_') for k in os.environ))
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python -c "import torch, PIL, sys; print('python', sys.version.split()[0], '| torch', torch.__version__, 'cuda', torch.version.cuda, 'avail', torch.cuda.is_available(), '| pillow', PIL.__version__)"


host: c5dc3075435a | Linux-6.6.122+-x86_64-with-glibc2.39
colab: True
name, memory.total [MiB], driver_version
NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07
python 3.13.15 | torch 2.11.0+cu130 cuda 13.0 avail True | pillow 11.3.0


## Cell 2: pinned install
Run once on a fresh runtime. Takes 1 to 3 minutes.

In [3]:
# Cell 2: pinned install. Run ONCE on a fresh runtime, before importing torch/transformers/unsloth.
# Never upgrades Colab's torch, torchvision, triton or Pillow: upgrading Pillow inside a live kernel
# is what caused "_imaging built for another version" (error A) and torchvision's
# "pic should be PIL Image. Got PIL.Image.Image" (error B).
import importlib.metadata as md, subprocess, sys

PROTECTED = ("torch", "torchvision", "triton", "pillow")
before = {name: md.version(name) for name in PROTECTED}

NO_DEPS = [  # versions released on or before 2026-10-03; Unsloth's own Colab recipe uses --no-deps
    "unsloth==2026.9.14", "unsloth_zoo==2026.9.9",
    "trl==0.22.2", "peft==0.21.1", "accelerate==1.15.0",
    "bitsandbytes==0.50.2", "torchao==0.18.0", "cut_cross_entropy==25.1.1",
    "https://download.pytorch.org/whl/cu130/xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl",
]
WITH_DEPS = [  # pure-Python; their dependencies never include torch or Pillow
    "transformers==4.57.6", "datasets==4.3.0",
    "sentencepiece==0.2.1", "tyro==1.0.16", "msgspec==0.21.1", "hf_transfer==0.1.9",
]
def pip(*args):
    print("$ pip install", " ".join(args))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("--no-deps", *NO_DEPS)
pip(*WITH_DEPS)

after = {name: md.version(name) for name in PROTECTED}
changed = {k: (before[k], after[k]) for k in PROTECTED if before[k] != after[k]}
if changed:
    raise RuntimeError(f"install changed protected packages {changed}; restart the runtime and report this")
print("Protected packages unchanged:", after)
print("Install finished. Run cell 3 next (no restart needed on a fresh runtime).")


$ pip install --no-deps unsloth==2026.9.14 unsloth_zoo==2026.9.9 trl==0.22.2 peft==0.21.1 accelerate==1.15.0 bitsandbytes==0.50.2 torchao==0.18.0 cut_cross_entropy==25.1.1 https://download.pytorch.org/whl/cu130/xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl
$ pip install transformers==4.57.6 datasets==4.3.0 sentencepiece==0.2.1 tyro==1.0.16 msgspec==0.21.1 hf_transfer==0.1.9
Protected packages unchanged: {'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'triton': '3.6.0', 'pillow': '11.3.0'}
Install finished. Run cell 3 next (no restart needed on a fresh runtime).


## Cell 3: environment check
Must print `ENVIRONMENT OK` before any training code runs.

In [4]:
# Cell 3: verify the environment before any training code runs.
import unsloth  # must be imported before transformers/trl/peft so its patches apply
import importlib.metadata as md, sys
import PIL, PIL.Image, torch, torchvision, transformers, trl, peft, datasets, accelerate, bitsandbytes
from torchvision.transforms import functional as TF

EXPECTED = {"unsloth": "2026.9.14", "unsloth_zoo": "2026.9.9", "transformers": "4.57.6",
            "trl": "0.22.2", "peft": "0.21.1", "accelerate": "1.15.0", "datasets": "4.3.0",
            "bitsandbytes": "0.50.2"}
for name, want in EXPECTED.items():
    got = md.version(name)
    assert got == want, f"{name} is {got}, expected {want}: restart the runtime and rerun cell 2"

# Errors A/B: the Pillow module in memory must be the installed one, and torchvision must accept its images.
assert PIL.__version__ == md.version("pillow"), (
    f"Pillow in memory {PIL.__version__} != installed {md.version('pillow')}: Runtime > Restart session")
tensor = TF.pil_to_tensor(PIL.Image.new("RGB", (4, 4)))
assert tensor.shape == (3, 4, 4)

# Error D: transformers 4.57 still accepts warmup_ratio; we pass integer warmup_steps anyway.
from trl import SFTConfig
SFTConfig(output_dir="/tmp/probe", warmup_steps=1, report_to="none")

assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()
props = torch.cuda.get_device_properties(0)
print(f"GPU {props.name} {props.total_memory/2**30:.1f} GiB | torch {torch.__version__} CUDA {torch.version.cuda}")
print("Pillow", PIL.__version__, PIL.__file__)
print({name: md.version(name) for name in [*EXPECTED, "torchvision", "torchao", "xformers", "huggingface-hub", "tokenizers"]})
print("ENVIRONMENT OK")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU NVIDIA A100-SXM4-40GB 39.5 GiB | torch 2.11.0+cu130 CUDA 13.0
Pillow 11.3.0 /usr/local/lib/python3.13/dist-packages/PIL/__init__.py
{'unsloth': '2026.9.14', 'unsloth_zoo': '2026.9.9', 'transformers': '4.57.6', 'trl': '0.22.2', 'peft': '0.21.1', 'accelerate': '1.15.0', 'datasets': '4.3.0', 'bitsandbytes': '0.50.2', 'torchvision': '0.26.0+cu130', 'torchao': '0.18.0', 'xformers': '0.0.35', 'huggingface-hub': '0.36.2', 'tokenizers': '0.22.2'}
ENVIRONMENT OK


## Execution order
Run cells top to bottom (**Run All** is safe). Each stage has a gate:

| Cell | Stage | Stops the run if |
|---|---|---|
| 1-3 | Location, pinned install, environment check | wrong GPU, any version mismatch, Pillow/torchvision mismatch |
| 4-6 | Engine module, settings, CORD data and schema checks | invalid config, bad rows, any example over `max_seq_len` |
| 7 | **Skill baseline**: 4-bit base model with the short prompt vs with the whole receipt skill, 100 validation receipts | skill files differ from the reviewed contract |
| 8 | GPU compatibility tests | collator masking, gradients, frozen weights or save/reload is wrong |
| 9 | Smoke test (12 receipts, 5 steps) | non-finite loss or no answer tokens supervised |
| 10 | Microbatch benchmark (1, 2, 4 at effective batch 8) | nothing completes stably |
| 11 | Hyperparameter trials (3 trials, 256 train, 20 validation, 35 steps) | |
| 12 | Recommendation and **approval gate** | full training runs only after you set `APPROVE_FULL_TRAINING = True` |
| 13-16 | Full training, evaluation vs 4-bit control and BF16 baseline, final test, export | skipped until approved |

Progress is printed as plain text so it stays readable in the saved notebook. Every stage also writes JSON under `/content/dapta_lora/`.

## Cell 4: engine module
The tested training engine, written verbatim from `notebooks/dapta_vlm_engine.py` (sha `f5b60903a68c`).

In [ ]:
%%writefile /content/dapta_vlm_engine.py
"""DAPTA VLM QLoRA engine: Qwen3-VL document extraction with Unsloth.

The notebook is the interface; this module holds the stable logic so every stage
(tests, smoke, benchmark, tuning, full training, evaluation) uses the same code.
GPU-only dependencies (torch, unsloth, trl) are imported inside the functions that
need them, so data, scoring, masking audits and selection logic run anywhere.
"""

from __future__ import annotations

import gc
import json
import math
import random
import re
import statistics
import time
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Iterable, Mapping, Sequence

# The baseline's short extraction prompt. Changing it breaks comparability with the
# zero-shot baseline, so it is fixed here rather than in the notebook settings.
PROMPT = (
    "Extract this receipt as JSON. Use CORD-style keys: menu (nm, cnt, "
    "unitprice, price), sub_total (subtotal_price, tax_price), and "
    "total (total_price, cashprice, changeprice). "
    "Include only fields visible on the receipt. Return only JSON."
)
INSTRUCTION_PART = "<|im_start|>user\n"
RESPONSE_PART = "<|im_start|>assistant\n"
END_OF_TURN = "<|im_end|>"
VISUAL_TOKENS = ("<|image_pad|>", "<|vision_start|>", "<|vision_end|>")
BASELINE_BF16 = {  # zero-shot, BF16, all 100 CORD validation receipts, same scorer and prompt
    "field_f1": 0.443793,
    "field_precision": 0.400794,
    "field_recall": 0.497129,
    "valid_json_fraction": 1.0,
    "total_price_exact_fraction": 0.387755,
    "mean_seconds_per_receipt": 9.918,
    "mean_input_tokens": 1201.33,
    "mean_output_tokens": 198.64,
}


# --------------------------------------------------------------------------- config


@dataclass(frozen=True)
class EngineConfig:
    model_id: str = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"
    dataset_id: str = "naver-clova-ix/cord-v2"
    # Pinned so every run sees the same 800/100/100 receipts.
    dataset_revision: str = "7f0115a4b758a71d6473b8d085751692da2fef98"
    prompt: str = PROMPT
    seed: int = 42
    image_long_side: int = 1280
    # Unsloth defaults to 2048 and its collator then silently truncates; every example
    # is length-checked against this limit before training instead.
    max_seq_len: int = 4096
    max_new_tokens: int = 1536
    root: str = "/content/dapta_lora"

    def validate(self) -> "EngineConfig":
        _require(self.prompt == PROMPT, "prompt must stay the baseline's short prompt")
        _require(256 <= self.image_long_side <= 4096, "image_long_side must be 256..4096")
        _require(1024 <= self.max_seq_len <= 32768, "max_seq_len must be 1024..32768")
        _require(64 <= self.max_new_tokens <= 8192, "max_new_tokens must be 64..8192")
        _require(re.fullmatch(r"[0-9a-f]{40}", self.dataset_revision) is not None,
                 "dataset_revision must be a full commit sha")
        return self


@dataclass(frozen=True)
class TrainSpec:
    name: str
    rank: int
    lr: float
    batch: int
    accum: int
    steps: int | None = None
    epochs: float | None = None
    warmup_fraction: float = 0.05
    weight_decay: float = 0.01
    eval_every: int | None = None
    save_every: int | None = None
    early_stop_patience: int | None = None
    log_every: int = 1

    def validate(self) -> "TrainSpec":
        _require(re.fullmatch(r"[A-Za-z0-9_.-]{1,64}", self.name) is not None, "invalid run name")
        _require(self.rank in (4, 8, 16, 32, 64), "rank must be one of 4, 8, 16, 32, 64")
        _require(0 < self.lr <= 1e-3, "lr must be in (0, 1e-3]")
        _require(self.batch >= 1 and self.accum >= 1, "batch and accum must be >= 1")
        _require((self.steps is None) != (self.epochs is None), "set exactly one of steps or epochs")
        _require(self.steps is None or self.steps >= 1, "steps must be >= 1")
        _require(self.epochs is None or 0 < self.epochs <= 10, "epochs must be in (0, 10]")
        _require(0 <= self.warmup_fraction < 0.5, "warmup_fraction must be in [0, 0.5)")
        _require(self.early_stop_patience is None or self.eval_every is not None,
                 "early stopping needs eval_every")
        return self

    @property
    def effective_batch(self) -> int:
        return self.batch * self.accum

    def total_steps(self, train_size: int) -> int:
        if self.steps is not None:
            return self.steps
        return math.ceil(train_size / self.effective_batch) * math.ceil(self.epochs)

    def warmup_steps(self, train_size: int) -> int:
        # Integer steps: transformers 5 renamed warmup_ratio, and integers mean the same everywhere.
        return max(1, math.ceil(self.warmup_fraction * self.total_steps(train_size)))


def _require(condition: bool, message: str) -> None:
    if not condition:
        raise ValueError(message)


def set_seed(seed: int) -> None:
    random.seed(seed)
    try:
        from transformers import set_seed as hf_set_seed
    except ImportError:
        return
    hf_set_seed(seed)


# --------------------------------------------------------------------------- data


def gold_parse(row: Mapping[str, Any]) -> dict[str, Any]:
    value = json.loads(row["ground_truth"])["gt_parse"]
    if not isinstance(value, dict):
        raise TypeError(f"gt_parse must be a JSON object, got {type(value).__name__}")
    return value


def target_text(gold: Mapping[str, Any]) -> str:
    """The exact training answer: compact JSON that must parse back to the gold object."""
    text = json.dumps(gold, ensure_ascii=False, separators=(",", ":"))
    if json.loads(text) != gold:
        raise ValueError("training target does not round-trip through JSON")
    return text


def prepare_image(image: Any, long_side: int) -> Any:
    """RGB copy with the long side capped; the same function serves training and evaluation."""
    from PIL import Image

    if not isinstance(image, Image.Image):
        raise TypeError(f"expected a PIL image, got {type(image).__module__}.{type(image).__name__}")
    image = image.convert("RGB")
    if max(image.size) > long_side:
        image = image.copy()
        image.thumbnail((long_side, long_side), Image.Resampling.LANCZOS)
    return image


def build_messages(image: Any, prompt: str, answer: str | None = None) -> list[dict[str, Any]]:
    """Qwen3-VL conversation: image then the short prompt, optionally the assistant answer."""
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ]}]
    if answer is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return messages


def require_receipt_schema(split: Any) -> None:
    """CORD rows must carry a typed Image feature and a string ground_truth column."""
    from datasets import Image as ImageFeature, Value

    features = split.features
    _require("image" in features and "ground_truth" in features, f"expected image/ground_truth columns, got {list(features)}")
    if not isinstance(features["image"], ImageFeature):
        raise TypeError(f"'image' must be a datasets.Image feature, got {features['image']!r}; "
                        "use split.cast_column('image', datasets.Image())")
    _require(features["ground_truth"] == Value("string"), "'ground_truth' must be a string column")


def receipt_dataset(split: Any, indices: Sequence[int], cfg: EngineConfig) -> Any:
    """An HF Dataset (TRL accepts no other type) that builds conversations lazily on access."""
    from PIL import Image

    require_receipt_schema(split)
    subset = split.select(list(indices))
    # with_transform receives the stored record ({"bytes", "path"}), not a decoded image;
    # decode it through the split's own Image feature so the type is always a PIL image.
    image_feature = split.features["image"]

    def to_messages(batch: Mapping[str, list[Any]]) -> dict[str, list[Any]]:
        conversations = []
        for stored, ground_truth in zip(batch["image"], batch["ground_truth"]):
            image = stored if isinstance(stored, Image.Image) else image_feature.decode_example(stored)
            answer = target_text(gold_parse({"ground_truth": ground_truth}))
            conversations.append(build_messages(prepare_image(image, cfg.image_long_side), cfg.prompt, answer))
        return {"messages": conversations}

    return subset.with_transform(to_messages)


def check_rows(split: Any, indices: Iterable[int]) -> dict[str, Any]:
    """Schema check of raw rows: gold JSON object, round-trip target, decodable RGB-able image."""
    require_receipt_schema(split)
    checked, failures = 0, []
    for index in indices:
        row = split[int(index)]
        try:
            target_text(gold_parse(row))
            row["image"].convert("RGB")
            checked += 1
        except (TypeError, ValueError, KeyError, OSError, json.JSONDecodeError) as error:
            failures.append({"index": int(index), "error": f"{type(error).__name__}: {error}"})
    if failures:
        raise ValueError(f"{len(failures)} rows failed schema checks: {failures[:5]}")
    return {"rows_checked": checked}


def sequence_lengths(processor: Any, dataset: Any) -> list[int]:
    """Exact token count of each training conversation, image tokens included."""
    lengths = []
    for index in range(len(dataset)):
        messages = dataset[index]["messages"]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        image = messages[0]["content"][0]["image"]
        batch = processor(text=[text], images=[image], return_tensors="pt", add_special_tokens=False)
        lengths.append(int(batch["input_ids"].shape[-1]))
    return lengths


def length_report(lengths: Sequence[int], limit: int) -> dict[str, Any]:
    ordered = sorted(lengths)
    report = {
        "examples": len(ordered),
        "min": ordered[0],
        "median": ordered[len(ordered) // 2],
        "p95": ordered[min(len(ordered) - 1, math.ceil(0.95 * len(ordered)) - 1)],
        "max": ordered[-1],
        "limit": limit,
        "over_limit": sum(value > limit for value in ordered),
    }
    if report["over_limit"]:
        raise ValueError(f"{report['over_limit']} examples exceed max_seq_len {limit}; raise it or drop them")
    return report


# --------------------------------------------------------------------------- masking audit


def audit_batch(batch: Mapping[str, Any], tokenizer: Any, answers: Sequence[str]) -> dict[str, int]:
    """Prove the loss sees exactly each answer + end-of-turn, and never prompt or image tokens."""
    import torch

    input_ids, labels = batch["input_ids"], batch["labels"]
    _require(input_ids.shape == labels.shape, "labels must align with input_ids")
    _require(len(answers) == input_ids.shape[0], "one expected answer per row")
    visual_ids = [tokenizer.convert_tokens_to_ids(token) for token in VISUAL_TOKENS]
    visual = torch.isin(input_ids, torch.tensor(visual_ids, device=input_ids.device))
    _require(int(visual.sum()) > 0, "batch contains no visual tokens: images did not reach the processor")
    _require(not bool((labels[visual] != -100).any()), "visual tokens are supervised")
    if "attention_mask" in batch:
        padding = batch["attention_mask"] == 0
        _require(not bool((labels[padding] != -100).any()), "padding tokens are supervised")

    supervised_total = 0
    for row, answer in enumerate(answers):
        kept = labels[row] != -100
        _require(int(kept.sum()) > 0, f"row {row} has no supervised tokens")
        decoded = tokenizer.decode(input_ids[row][kept], skip_special_tokens=False)
        _require(decoded.startswith(answer), f"row {row} supervises text other than the answer: {decoded[:120]!r}")
        tail = decoded[len(answer):]
        _require(tail.strip() == END_OF_TURN, f"row {row} supervised tail is {tail!r}, expected {END_OF_TURN!r}")
        positions = kept.nonzero().flatten()
        _require(bool((positions[1:] - positions[:-1] == 1).all()), f"row {row} supervision is not contiguous")
        supervised_total += int(kept.sum())
    return {
        "rows": len(answers),
        "supervised_tokens": supervised_total,
        "ignored_tokens": int((labels == -100).sum()),
        "visual_tokens": int(visual.sum()),
        "sequence_length": int(input_ids.shape[1]),
    }


# --------------------------------------------------------------------------- scorer
# Unchanged from the baseline notebook: flattened (field path, normalized value) pairs,
# position-sensitive menu indices. Not the official CORD metric.


def parse_json_object(text):
    raw = text.strip()
    for s in (raw, re.sub(r"^```(?:json)?\s*|\s*```$", "", raw, flags=re.I)):
        try:
            obj = json.loads(s.strip())
            if isinstance(obj, dict): return obj
        except (json.JSONDecodeError, ValueError): pass
    decoder = json.JSONDecoder()
    for m in re.finditer(r"\{", raw):
        try:
            obj, _ = decoder.raw_decode(raw[m.start():])
            if isinstance(obj, dict): return obj
        except json.JSONDecodeError: pass
    return None


def canonical(obj):
    if isinstance(obj, dict) and isinstance(obj.get("gt_parse"), dict): obj = obj["gt_parse"]
    if not isinstance(obj, dict): return {}
    obj = dict(obj)
    if isinstance(obj.get("menu"), dict): obj["menu"] = [obj["menu"]]
    return obj


def norm(v): return str(v).lower() if isinstance(v, bool) else " ".join(str(v).strip().casefold().split())


def flatten(obj, prefix=""):
    if isinstance(obj, dict):
        return [item for k, v in sorted(obj.items()) for item in flatten(v, f"{prefix}.{k}" if prefix else k)]
    if isinstance(obj, list):
        return [item for i, v in enumerate(obj) for item in flatten(v, f"{prefix}[{i}]")]
    return [] if obj is None else [(prefix, norm(obj))]


def score_example(pred, gold):
    p, g = canonical(pred), canonical(gold)
    a, b = Counter(flatten(p)), Counter(flatten(g))
    gt_total = g.get("total") if isinstance(g.get("total"), dict) else {}
    pr_total = p.get("total") if isinstance(p.get("total"), dict) else {}
    gold_price, pred_price = gt_total.get("total_price"), pr_total.get("total_price")
    return {"tp": sum((a & b).values()), "pred_fields": sum(a.values()), "gold_fields": sum(b.values()),
            "document_exact": a == b, "total_present": gold_price is not None,
            "total_exact": gold_price is not None and pred_price is not None and norm(gold_price) == norm(pred_price)}


def summarize(rows):
    tp = sum(r["score"]["tp"] for r in rows)
    npred = sum(r["score"]["pred_fields"] for r in rows)
    ngold = sum(r["score"]["gold_fields"] for r in rows)
    p = tp / npred if npred else 0.
    r = tp / ngold if ngold else 0.
    totals = [x for x in rows if x["score"]["total_present"]]
    n = len(rows)
    return {"samples": n, "field_precision": round(p, 6), "field_recall": round(r, 6),
            "field_f1": round(2*p*r/(p+r) if p+r else 0., 6),
            "valid_json_fraction": round(sum(x["valid_json"] for x in rows)/n, 6) if n else 0,
            "document_exact_fraction": round(sum(x["score"]["document_exact"] for x in rows)/n, 6) if n else 0,
            "total_price_exact_fraction": round(sum(x["score"]["total_exact"] for x in totals)/len(totals), 6) if totals else None,
            "mean_seconds_per_receipt": round(statistics.mean(x["elapsed_s"] for x in rows), 3) if n else None,
            "mean_input_tokens": round(statistics.mean(x["input_tokens"] for x in rows), 2) if n else None,
            "mean_output_tokens": round(statistics.mean(x["output_tokens"] for x in rows), 2) if n else None}


# --------------------------------------------------------------------------- receipt-to-expense skill
# Grading for answers in the repo's receipt-to-expense skill format, following that skill's
# evaluation_plan.json (cord_field_accuracy): line items match on normalized description and
# amount within 0.01, scalar amounts within 0.01. CORD gold is converted to the same format.

SKILL_SYSTEM_PREFIX = "Extract visible receipt evidence and return only JSON matching the receipt expense schema."
SKILL_USER_PROMPT = "Convert this receipt image into the normalized expense report. Use null for absent fields."
EXPENSE_SCALARS = ("subtotal", "tax", "service_charge", "total")
AMOUNT_ABS = 0.01


def full_skill_prompt(skill_spec: Mapping[str, Any], output_schema: Mapping[str, Any]) -> str:
    """The whole skill document, unchanged, as the system prompt."""
    return (f"{SKILL_SYSTEM_PREFIX}\n\nFollow this skill exactly.\n\nSKILL\n"
            f"{json.dumps(skill_spec, indent=2, ensure_ascii=False)}\n\nOUTPUT JSON SCHEMA\n"
            f"{json.dumps(output_schema, indent=2, ensure_ascii=False)}")


def parse_amount(value: Any) -> float | None:
    """Printed amount to a number. CORD receipts are Indonesian: '18.000' is eighteen thousand."""
    from decimal import Decimal, InvalidOperation

    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, (list, tuple)):
        parsed = [p for p in (parse_amount(v) for v in value) if p is not None]
        return parsed[0] if parsed else None
    text = re.sub(r"[^0-9,.\-]", "", str(value).strip())
    if not re.search(r"\d", text):
        return None
    if "," in text and "." in text:
        decimal = "." if text.rfind(".") > text.rfind(",") else ","
        text = text.replace("," if decimal == "." else ".", "").replace(decimal, ".")
    elif "," in text:
        text = text.replace(",", "") if re.fullmatch(r"-?\d{1,3}(,\d{3})+", text) else text.replace(",", ".")
    elif re.fullmatch(r"-?\d{1,3}(\.\d{3})+", text):
        text = text.replace(".", "")
    try:
        return float(Decimal(text))
    except InvalidOperation:
        return None


def _text_or_none(value: Any) -> str | None:
    if value is None:
        return None
    text = " ".join(str(value).split())
    return text or None


def cord_to_expense(obj: Any) -> dict[str, Any]:
    """CORD gt_parse (gold, or a CORD-format prediction) in the skill's expense-receipt shape."""
    obj = canonical(obj)
    menu = obj.get("menu") if isinstance(obj.get("menu"), list) else []
    sub = obj.get("sub_total") if isinstance(obj.get("sub_total"), dict) else {}
    total = obj.get("total") if isinstance(obj.get("total"), dict) else {}
    return {
        "merchant": None, "address": None, "transaction_date": None, "currency": None,
        "subtotal": parse_amount(sub.get("subtotal_price")),
        "tax": parse_amount(sub.get("tax_price")),
        "service_charge": parse_amount(sub.get("service_price")),
        "total": parse_amount(total.get("total_price")),
        "line_items": [
            {"description": _text_or_none(item.get("nm")),
             "quantity": parse_amount(item.get("cnt")),
             "amount": parse_amount(item.get("price"))}
            for item in menu if isinstance(item, dict)
        ],
    }


def _same_amount(a: Any, b: Any) -> bool:
    if a is None or b is None:
        return a is None and b is None
    if not isinstance(a, (int, float)) or not isinstance(b, (int, float)) or isinstance(a, bool) or isinstance(b, bool):
        return False
    return abs(float(a) - float(b)) <= AMOUNT_ABS


def _description(value: Any) -> str | None:
    return None if value is None else (" ".join(str(value).casefold().split()) or None)


def score_expense(prediction: Any, gold_receipt: Mapping[str, Any], validator: Any) -> dict[str, Any]:
    """One receipt. A schema-invalid answer scores zero (the skill plan's gate) but its gold still counts."""
    schema_valid = (isinstance(prediction, dict) and not any(True for _ in validator.iter_errors(prediction))
                    and len(prediction.get("receipts", [])) == 1)
    receipt = prediction["receipts"][0] if schema_valid else None
    gold_items = [(_description(i["description"]), i["amount"], i["quantity"]) for i in gold_receipt["line_items"]]
    pred_items = ([(_description(i.get("description")), i.get("amount"), i.get("quantity")) for i in receipt["line_items"]]
                  if receipt else [])
    used = [False] * len(pred_items)
    items_tp = 0
    for description, amount, quantity in gold_items:
        for j, (p_desc, p_amount, p_quantity) in enumerate(pred_items):
            if (not used[j] and p_desc == description and _same_amount(p_amount, amount)
                    and (quantity is None or _same_amount(p_quantity, quantity))):
                used[j] = True
                items_tp += 1
                break
    scalars = {}
    for field in EXPENSE_SCALARS:
        gold_value = gold_receipt[field]
        pred_value = receipt.get(field) if receipt else None
        scalars[field] = {"gold": gold_value is not None, "pred": pred_value is not None,
                          "correct": gold_value is not None and _same_amount(pred_value, gold_value)}
    return {"schema_valid": schema_valid, "items_tp": items_tp, "items_pred": len(pred_items),
            "items_gold": len(gold_items), "scalars": scalars,
            "currency_set": bool(receipt and receipt.get("currency") is not None)}


def summarize_expense(rows: Sequence[Mapping[str, Any]], *, schema_measured: bool = True) -> dict[str, Any]:
    scores = [r["expense"] for r in rows]
    n = len(scores)

    def prf(tp: int, pred: int, gold: int) -> tuple[float, float, float]:
        p = tp / pred if pred else 0.0
        r = tp / gold if gold else 0.0
        return round(p, 6), round(r, 6), round(2 * p * r / (p + r) if p + r else 0.0, 6)

    items = prf(sum(s["items_tp"] for s in scores), sum(s["items_pred"] for s in scores), sum(s["items_gold"] for s in scores))
    report: dict[str, Any] = {
        "samples": n,
        "schema_valid_fraction": round(sum(s["schema_valid"] for s in scores) / n, 6) if schema_measured and n else None,
        "line_item_precision": items[0], "line_item_recall": items[1], "line_item_f1": items[2],
    }
    tp, pred, gold = sum(s["items_tp"] for s in scores), sum(s["items_pred"] for s in scores), sum(s["items_gold"] for s in scores)
    for field in EXPENSE_SCALARS:
        labelled = [s["scalars"][field] for s in scores if s["scalars"][field]["gold"]]
        report[f"{field}_accuracy"] = round(sum(x["correct"] for x in labelled) / len(labelled), 6) if labelled else None
        report[f"{field}_unlabelled_but_predicted"] = sum(1 for s in scores if s["scalars"][field]["pred"] and not s["scalars"][field]["gold"])
        tp += sum(s["scalars"][field]["correct"] for s in scores)
        pred += sum(s["scalars"][field]["pred"] for s in scores)
        gold += sum(s["scalars"][field]["gold"] for s in scores)
    overall = prf(tp, pred, gold)
    report.update({"field_precision": overall[0], "field_recall": overall[1], "field_f1": overall[2],
                   "currency_set_count": sum(s["currency_set"] for s in scores)})
    for key in ("elapsed_s", "input_tokens", "output_tokens"):
        values = [r[key] for r in rows if key in r]
        report[f"mean_{'seconds_per_receipt' if key == 'elapsed_s' else key}"] = (
            round(statistics.mean(values), 3 if key == "elapsed_s" else 2) if values else None)
    report["valid_json_fraction"] = round(sum(r["valid_json"] for r in rows) / n, 6) if n else 0
    return report


def micro_f1(rows: Sequence[Mapping[str, Any]]) -> float:
    tp = sum(r["score"]["tp"] for r in rows)
    npred = sum(r["score"]["pred_fields"] for r in rows)
    ngold = sum(r["score"]["gold_fields"] for r in rows)
    p = tp / npred if npred else 0.0
    r = tp / ngold if ngold else 0.0
    return 2 * p * r / (p + r) if p + r else 0.0


def paired_bootstrap(a: Sequence[Mapping[str, Any]], b: Sequence[Mapping[str, Any]],
                     *, samples: int = 2000, seed: int = 42) -> dict[str, float]:
    """95% interval of micro-F1(a) - micro-F1(b) over the same receipts, resampled together."""
    by_index_b = {row["index"]: row for row in b}
    pairs = [(row, by_index_b[row["index"]]) for row in a if row["index"] in by_index_b]
    _require(len(pairs) == len(a) == len(b), "bootstrap needs predictions for the same receipts")
    rng = random.Random(seed)
    deltas = []
    for _ in range(samples):
        drawn = [pairs[rng.randrange(len(pairs))] for _ in pairs]
        deltas.append(micro_f1([x for x, _ in drawn]) - micro_f1([y for _, y in drawn]))
    deltas.sort()
    return {
        "delta": micro_f1(a) - micro_f1(b),
        "ci95_low": deltas[int(0.025 * samples)],
        "ci95_high": deltas[int(0.975 * samples) - 1],
    }


# --------------------------------------------------------------------------- selection


def choose_batch(rows: Sequence[Mapping[str, Any]], capacity_gib: float, headroom: float = 0.15) -> dict[str, Any]:
    """Fastest stable candidate whose peak reserved memory leaves `headroom` of the GPU free."""
    viable = [row for row in rows if not row.get("oom") and row.get("stable")]
    _require(bool(viable), "no microbatch completed stably; lower image_long_side and rerun")
    safe = [row for row in viable if row["peak_reserved_gib"] <= capacity_gib * (1 - headroom)]
    pool = safe or [min(viable, key=lambda row: row["peak_reserved_gib"])]
    best = max(pool, key=lambda row: row["examples_per_second"])
    return {
        "microbatch": best["microbatch"],
        "grad_accum": best["grad_accum"],
        "reason": ("fastest within memory headroom" if safe
                   else "no candidate within headroom; chose the lowest-memory stable candidate"),
        "headroom_fraction": headroom,
    }


def choose_trial(results: Sequence[Mapping[str, Any]], predictions: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Rank by validation field F1, then validation loss; flag when the top two are not separable."""
    ranked = sorted(results, key=lambda r: (-r["val_field_f1"], r["val_loss"]))
    best = dict(ranked[0])
    decision: dict[str, Any] = {"best": best, "ranking": [r["name"] for r in ranked], "conclusive": True}
    if len(ranked) > 1:
        runner = ranked[1]
        interval = paired_bootstrap(predictions[best["name"]], predictions[runner["name"]])
        decision["versus_runner_up"] = {"runner_up": runner["name"], **interval}
        if interval["ci95_low"] <= 0.0:
            decision["conclusive"] = False
            decision["recommendation"] = (
                f"{best['name']} and {runner['name']} are not separable on {len(predictions[best['name']])} "
                "validation receipts (95% interval of the F1 difference includes 0). Validate both on more "
                "receipts before treating either as best."
            )
    return decision


# --------------------------------------------------------------------------- GPU: model, collator, trainer


def gpu_memory() -> dict[str, float]:
    import torch

    gib = 2 ** 30
    return {
        "allocated_gib": round(torch.cuda.memory_allocated() / gib, 2),
        "reserved_gib": round(torch.cuda.memory_reserved() / gib, 2),
        "peak_allocated_gib": round(torch.cuda.max_memory_allocated() / gib, 2),
        "peak_reserved_gib": round(torch.cuda.max_memory_reserved() / gib, 2),
        "total_gib": round(torch.cuda.get_device_properties(0).total_memory / gib, 2),
    }


def free_gpu() -> dict[str, float]:
    """Collect garbage and empty the CUDA cache. Callers must drop their own references first."""
    import torch

    gc.collect()
    torch.cuda.empty_cache()
    return gpu_memory()


def load_base(cfg: EngineConfig) -> tuple[Any, Any]:
    from unsloth import FastVisionModel

    set_seed(cfg.seed)
    model, processor = FastVisionModel.from_pretrained(
        cfg.model_id,
        load_in_4bit=True,
        max_seq_length=cfg.max_seq_len,
        use_gradient_checkpointing="unsloth",
    )
    return model, processor


def load_lora(cfg: EngineConfig, rank: int) -> tuple[Any, Any]:
    """Fresh 4-bit base weights plus a newly initialised language-model LoRA."""
    from unsloth import FastVisionModel

    model, processor = load_base(cfg)
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=False,  # vision encoder stays frozen
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=int(rank), lora_alpha=int(rank), lora_dropout=0,
        bias="none", random_state=cfg.seed,
    )
    FastVisionModel.for_training(model)
    return model, processor


def trainable_report(model: Any) -> dict[str, Any]:
    trainable = [(name, p) for name, p in model.named_parameters() if p.requires_grad]
    total = sum(p.numel() for _, p in model.named_parameters())
    _require(bool(trainable), "no trainable parameters")
    bad = [name for name, _ in trainable if "lora_" not in name]
    _require(not bad, f"non-LoRA parameters are trainable: {bad[:5]}")
    vision = [name for name, _ in trainable if ".visual." in name or "vision_tower" in name]
    _require(not vision, f"vision-encoder LoRA found although vision layers are frozen: {vision[:5]}")
    count = sum(p.numel() for _, p in trainable)
    return {"trainable_parameters": count, "total_parameters": total,
            "trainable_fraction": round(count / total, 6), "trainable_tensors": len(trainable)}


def make_collator(model: Any, processor: Any, cfg: EngineConfig) -> Any:
    from unsloth.trainer import UnslothVisionDataCollator

    return UnslothVisionDataCollator(
        model, processor,
        max_seq_length=cfg.max_seq_len,  # explicit: the inherited default is 2048
        resize="max",  # no collator resize: Qwen3-VL has no image_size, so "min" fell back to 512px
        train_on_responses_only=True,
        instruction_part=INSTRUCTION_PART,
        response_part=RESPONSE_PART,
    )


def probe_collator(collator: Any, processor: Any, dataset: Any, rows: int = 2) -> dict[str, int]:
    """Collate real rows and audit the result; raises before any training step if masking is wrong."""
    examples = [dataset[i] for i in range(min(rows, len(dataset)))]
    batch = collator(examples)
    answers = [example["messages"][1]["content"][0]["text"] for example in examples]
    for key in ("input_ids", "attention_mask", "labels", "pixel_values", "image_grid_thw"):
        _require(key in batch, f"collator output is missing {key}")
    return audit_batch(batch, processor.tokenizer, answers)


def gradient_check(model: Any, collator: Any, dataset: Any) -> dict[str, Any]:
    """One forward/backward + optimizer step: LoRA moves, frozen weights stay bit-identical."""
    import torch

    model.train()
    batch = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in collator([dataset[0]]).items()}
    trainable = {n: p for n, p in model.named_parameters() if p.requires_grad}
    all_frozen = [(n, p) for n, p in model.named_parameters() if not p.requires_grad]
    frozen = all_frozen[:: max(1, len(all_frozen) // 16)][:16]
    frozen_before = {n: p.detach().float().flatten()[:4096].clone().cpu() for n, p in frozen}
    lora_before = {n: p.detach().float().clone().cpu() for n, p in list(trainable.items())[:16]}

    loss = model(**batch).loss
    _require(torch.isfinite(loss).item(), f"loss is not finite: {loss.item()}")
    loss.backward()
    with_grad = [n for n, p in trainable.items() if p.grad is not None and float(p.grad.abs().sum()) > 0]
    _require(bool(with_grad), "no LoRA parameter received a gradient")
    _require(all(p.grad is None for _, p in frozen), "a frozen parameter received a gradient")
    optimizer = torch.optim.AdamW(list(trainable.values()), lr=1e-4)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    lora_changed = sum(not torch.equal(lora_before[n], trainable[n].detach().float().cpu()) for n in lora_before)
    frozen_changed = [n for n, p in frozen if not torch.equal(frozen_before[n], p.detach().float().flatten()[:4096].cpu())]
    _require(lora_changed > 0, "optimizer step did not change any sampled LoRA tensor")
    _require(not frozen_changed, f"frozen parameters changed: {frozen_changed[:3]}")
    return {"loss": round(float(loss.item()), 4), "lora_tensors_with_grad": len(with_grad),
            "lora_tensors_total": len(trainable), "sampled_lora_changed": lora_changed,
            "sampled_frozen_unchanged": len(frozen)}


def lora_state(model: Any) -> dict[str, Any]:
    return {n.split("base_model.model.")[-1]: p.detach().cpu().clone()
            for n, p in model.named_parameters() if "lora_" in n}


def save_reload_check(model: Any, processor: Any, cfg: EngineConfig, out_dir: Path) -> dict[str, Any]:
    """Save the adapter, reload it on fresh 4-bit base weights, and compare LoRA tensors exactly."""
    import torch
    from unsloth import FastVisionModel

    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(out_dir))
    processor.save_pretrained(str(out_dir))
    files = sorted(p.name for p in out_dir.iterdir())
    _require("adapter_config.json" in files and "adapter_model.safetensors" in files,
             f"adapter files missing after save: {files}")
    before = lora_state(model)
    reloaded, _ = FastVisionModel.from_pretrained(str(out_dir), load_in_4bit=True, max_seq_length=cfg.max_seq_len)
    after = lora_state(reloaded)
    del reloaded
    free_gpu()
    _require(set(before) == set(after) and after, f"reloaded {len(after)} LoRA tensors, saved {len(before)}")
    # Compare in the reloaded dtype: a save/load may legitimately change float32 <-> bfloat16.
    mismatched = [n for n in before if not torch.equal(before[n].to(after[n].dtype), after[n])]
    _require(not mismatched, f"{len(mismatched)} LoRA tensors differ after reload: {mismatched[:3]}")
    dtypes = sorted({str(v.dtype) for v in after.values()})
    return {"adapter_dir": str(out_dir), "files": files, "lora_tensors_compared": len(before), "dtypes": dtypes}


class LiveProgress:
    """Plain-text progress (readable in saved notebook output) plus a JSONL history on disk."""

    def __init__(self, name: str, history_path: Path, log_every: int = 1):
        self.name = name
        self.history_path = history_path
        self.log_every = max(1, log_every)
        self.started = None
        self.last_step_end = None
        self.step_seconds: list[float] = []
        self.last_eval_loss = None
        self.utilization: list[int] = []

    def callback(self) -> Any:
        from transformers import TrainerCallback

        progress = self

        class _Callback(TrainerCallback):
            def on_train_begin(self, args, state, control, **kwargs):
                progress.started = progress.last_step_end = time.perf_counter()
                progress.history_path.parent.mkdir(parents=True, exist_ok=True)

            def on_step_end(self, args, state, control, **kwargs):
                # on_step_end fires once per optimizer step; the gap between calls is the step time.
                now = time.perf_counter()
                progress.step_seconds.append(now - progress.last_step_end)
                progress.last_step_end = now
                sample = _gpu_utilization()
                if sample is not None:
                    progress.utilization.append(sample)

            def on_log(self, args, state, control, logs=None, **kwargs):
                if not logs:
                    return
                if "eval_loss" in logs:
                    progress.last_eval_loss = logs["eval_loss"]
                record = {"step": state.global_step, "epoch": state.epoch, "time": time.time(), **logs, **gpu_memory()}
                with progress.history_path.open("a", encoding="utf-8") as handle:
                    handle.write(json.dumps(record) + "\n")
                if state.global_step % progress.log_every == 0 or "eval_loss" in logs or state.global_step == state.max_steps:
                    print(progress.format_line(state, logs), flush=True)

        return _Callback()

    def format_line(self, state: Any, logs: Mapping[str, Any]) -> str:
        elapsed = time.perf_counter() - (self.started or time.perf_counter())
        done, total = state.global_step, max(1, state.max_steps)
        eta = elapsed / done * (total - done) if done else float("nan")
        memory = gpu_memory()
        epochs = state.num_train_epochs if getattr(state, "num_train_epochs", None) else float("nan")
        loss = logs.get("loss")
        parts = [
            f"[{self.name}] Training: {done}/{total} steps",
            f"Progress: {100 * done / total:.1f}%",
            f"Epoch: {state.epoch or 0:.2f}/{epochs:.0f}" if epochs == epochs else f"Epoch: {state.epoch or 0:.2f}",
            f"Loss: {loss:.4f}" if loss is not None else "Loss: -",
            f"Validation loss: {self.last_eval_loss:.4f}" if self.last_eval_loss is not None else "Validation loss: -",
            f"Learning rate: {logs['learning_rate']:.1e}" if "learning_rate" in logs else "Learning rate: -",
            f"GPU memory: {memory['allocated_gib']:.1f}/{memory['total_gib']:.1f} GB (peak reserved {memory['peak_reserved_gib']:.1f})",
            f"Elapsed: {_clock(elapsed)}",
            f"ETA: {_clock(eta)}",
        ]
        return " | ".join(parts)


def _gpu_utilization() -> int | None:
    try:
        import torch

        return int(torch.cuda.utilization())
    except (ImportError, RuntimeError, ModuleNotFoundError):
        return None  # pynvml missing: utilization is reported as unavailable, not guessed


def _clock(seconds: float) -> str:
    if seconds != seconds or seconds == float("inf"):
        return "-"
    minutes, secs = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h {minutes:02d}m {secs:02d}s" if hours else f"{minutes}m {secs:02d}s"


def make_trainer(model: Any, processor: Any, collator: Any, train_ds: Any, spec: TrainSpec,
                 cfg: EngineConfig, out_dir: Path, *, val_ds: Any = None, save: bool = False,
                 progress: LiveProgress | None = None) -> Any:
    from transformers import EarlyStoppingCallback
    from trl import SFTConfig, SFTTrainer

    spec.validate()
    has_eval = val_ds is not None and spec.eval_every is not None
    keep_best = has_eval and save
    config = dict(
        output_dir=str(out_dir / "checkpoints"),
        per_device_train_batch_size=spec.batch,
        gradient_accumulation_steps=spec.accum,
        per_device_eval_batch_size=1,
        learning_rate=spec.lr,
        warmup_steps=spec.warmup_steps(len(train_ds)),
        lr_scheduler_type="cosine",
        weight_decay=spec.weight_decay,
        optim="adamw_8bit",
        max_grad_norm=1.0,
        bf16=True, fp16=False,
        logging_strategy="steps", logging_steps=1,
        report_to="none",
        eval_strategy="steps" if has_eval else "no",
        eval_steps=spec.eval_every if has_eval else None,
        save_strategy="steps" if save else "no",
        save_steps=spec.save_every if save else None,
        save_total_limit=2 if save else None,
        load_best_model_at_end=keep_best,
        metric_for_best_model="eval_loss" if keep_best else None,
        greater_is_better=False if keep_best else None,
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=None,  # the collator owns truncation; lengths were checked against max_seq_len
        dataloader_num_workers=0,
        seed=cfg.seed, data_seed=cfg.seed,
        disable_tqdm=True,
    )
    if spec.steps is not None:
        config["max_steps"] = spec.steps
    else:
        config["num_train_epochs"] = float(spec.epochs)
    callbacks = [progress.callback()] if progress else []
    if spec.early_stop_patience and keep_best:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=spec.early_stop_patience))
    return SFTTrainer(
        model=model, tokenizer=processor, data_collator=collator,
        train_dataset=train_ds, eval_dataset=val_ds if has_eval else None,
        args=SFTConfig(**config), callbacks=callbacks,
    )


def train_run(spec: TrainSpec, cfg: EngineConfig, train_ds: Any, *, val_ds: Any = None,
              save: bool = False, resume: bool = False) -> tuple[dict[str, Any], Any, Any, Any]:
    """Fresh weights, audited collator, training, and a metrics record. Caller releases the model."""
    import torch

    spec.validate()
    out_dir = Path(cfg.root) / spec.name
    out_dir.mkdir(parents=True, exist_ok=True)
    model = processor = trainer = collator = None
    try:
        torch.cuda.reset_peak_memory_stats()
        model, processor = load_lora(cfg, spec.rank)
        params = trainable_report(model)
        collator = make_collator(model, processor, cfg)
        mask = probe_collator(collator, processor, train_ds)
        progress = LiveProgress(spec.name, out_dir / "train_history.jsonl", spec.log_every)
        trainer = make_trainer(model, processor, collator, train_ds, spec, cfg, out_dir,
                               val_ds=val_ds, save=save, progress=progress)
        checkpoint = None
        if resume and (out_dir / "checkpoints").is_dir():
            from transformers.trainer_utils import get_last_checkpoint
            checkpoint = get_last_checkpoint(str(out_dir / "checkpoints"))
        started = time.perf_counter()
        stats = trainer.train(resume_from_checkpoint=checkpoint)
        seconds = time.perf_counter() - started
        history = [json.loads(line) for line in (out_dir / "train_history.jsonl").read_text().splitlines() if line.strip()]
        losses = [row["loss"] for row in history if "loss" in row]
        steady = progress.step_seconds[1:] or progress.step_seconds
        step_seconds = statistics.median(steady) if steady else float("nan")
        eval_loss = None
        if val_ds is not None:
            eval_loss = float(trainer.evaluate(eval_dataset=val_ds)["eval_loss"])
        if save:
            trainer.save_model(str(out_dir / "adapter"))
            processor.save_pretrained(str(out_dir / "adapter"))
        result = {
            "name": spec.name, **asdict(spec), "effective_batch": spec.effective_batch,
            "resumed_from": checkpoint, "steps_done": int(stats.global_step),
            "training_loss": float(stats.training_loss), "first_loss": losses[0] if losses else None,
            "last_loss": losses[-1] if losses else None, "val_loss": eval_loss,
            "stable": bool(losses) and all(math.isfinite(x) for x in losses),
            "train_seconds": round(seconds, 2), "median_step_seconds": round(step_seconds, 3),
            "examples_per_second": round(spec.effective_batch / step_seconds, 3) if step_seconds == step_seconds else None,
            "gpu_utilization_mean": round(statistics.mean(progress.utilization), 1) if progress.utilization else None,
            "mask_audit": mask, "parameters": params, **gpu_memory(),
        }
        (out_dir / "train_summary.json").write_text(json.dumps(result, indent=2))
        return result, model, processor, trainer
    except BaseException:
        # Drop this frame's references before re-raising so an OOM does not pin the model in memory.
        model = processor = trainer = collator = None
        free_gpu()
        raise


# --------------------------------------------------------------------------- evaluation


def evaluate_condition(model: Any, processor: Any, split: Any, indices: Sequence[int], cfg: EngineConfig,
                       *, desc: str, user_prompt: str, system_prompt: str | None = None,
                       answer_format: str = "cord", output_schema: Mapping[str, Any] | None = None,
                       output_dir: Path | None = None, print_every: int = 10) -> dict[str, Any]:
    """Greedy generation for one prompt condition; resumable from predictions.jsonl.

    answer_format "cord": CORD-keyed JSON, graded by the baseline scorer and, after the same
    conversion applied to gold, by the expense grader. "expense": the receipt-to-expense skill's
    format, graded by the expense grader with schema validation as the gate.
    """
    import torch
    from jsonschema import Draft202012Validator
    from unsloth import FastVisionModel

    _require(answer_format in ("cord", "expense"), "answer_format must be 'cord' or 'expense'")
    _require(answer_format == "cord" or output_schema is not None, "expense grading needs the output schema")
    validator = Draft202012Validator(output_schema) if output_schema is not None else None
    FastVisionModel.for_inference(model)
    path = None
    completed: dict[int, dict[str, Any]] = {}
    if output_dir:
        output_dir.mkdir(parents=True, exist_ok=True)
        path = output_dir / "predictions.jsonl"
        if path.exists():
            for line in path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    record = json.loads(line)
                    completed[int(record["index"])] = record
    torch.cuda.reset_peak_memory_stats()
    records = []
    started = time.perf_counter()
    for position, index in enumerate(indices, start=1):
        index = int(index)
        if index in completed:
            records.append(completed[index])
            continue
        row = split[index]
        gold = gold_parse(row)
        image = prepare_image(row["image"], cfg.image_long_side)
        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_prompt}]}]
        if system_prompt:
            messages.insert(0, {"role": "system", "content": [{"type": "text", "text": system_prompt}]})
        prompt_text = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(image, prompt_text, add_special_tokens=False, return_tensors="pt").to("cuda")
        input_length = int(inputs["input_ids"].shape[-1])
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.inference_mode():
            out = model.generate(**inputs, do_sample=False, use_cache=True, max_new_tokens=cfg.max_new_tokens)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        ids = out[0][input_length:]
        raw = processor.decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        parsed = parse_json_object(raw)
        gold_receipt = cord_to_expense(gold)
        record = {"index": index, "reference": gold, "prediction": parsed, "raw_prediction": raw,
                  "valid_json": parsed is not None, "elapsed_s": round(elapsed, 3),
                  "input_tokens": input_length, "output_tokens": int(ids.numel()),
                  "hit_max_new_tokens": int(ids.numel()) >= cfg.max_new_tokens}
        if answer_format == "cord":
            record["score"] = score_example(parsed, gold)
            if validator is not None:
                converted = {"receipts": [cord_to_expense(parsed)]} if parsed is not None else None
                record["expense"] = score_expense(converted, gold_receipt, validator)
        else:
            record["expense"] = score_expense(parsed, gold_receipt, validator)
        records.append(record)
        if path:
            with path.open("a", encoding="utf-8") as handle:
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        if position % print_every == 0 or position == len(indices):
            wall = time.perf_counter() - started
            now = summarize(records) if answer_format == "cord" else summarize_expense(records)
            print(f"[{desc}] {position}/{len(indices)} receipts | F1 {now['field_f1']:.3f} | "
                  f"valid JSON {now['valid_json_fraction']:.0%} | elapsed {_clock(wall)} | "
                  f"ETA {_clock(wall / position * (len(indices) - position))}", flush=True)
    if answer_format == "cord":
        summary = summarize(records)
        if validator is not None and all("expense" in r for r in records):
            summary["expense_format"] = summarize_expense(records, schema_measured=False)
    else:
        summary = summarize_expense(records)
    summary["truncated_outputs"] = sum(r.get("hit_max_new_tokens", False) for r in records)
    summary["peak_allocated_gib"] = gpu_memory()["peak_allocated_gib"]
    if output_dir:
        (output_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    FastVisionModel.for_training(model)
    return {"summary": summary, "records": records}


def evaluate_receipts(model: Any, processor: Any, split: Any, indices: Sequence[int], cfg: EngineConfig,
                      *, desc: str, output_dir: Path | None = None, print_every: int = 10) -> dict[str, Any]:
    """The baseline's short prompt, CORD format, baseline scorer."""
    return evaluate_condition(model, processor, split, indices, cfg, desc=desc, user_prompt=cfg.prompt,
                              answer_format="cord", output_dir=output_dir, print_every=print_every)


def compare(report: Mapping[str, Any], reference: Mapping[str, Any]) -> dict[str, Any]:
    keys = [k for k in BASELINE_BF16 if k in report and report[k] is not None and reference.get(k) is not None]
    return {k: {"run": report[k], "reference": reference[k], "delta": round(report[k] - reference[k], 6)} for k in keys}


def persist_to_hub(root: Path, repo_id: str, token: str | None, run_id: str, patterns: Sequence[str],
                   message: str) -> str | None:
    """Copy finished results off the Colab machine, which loses /content when the runtime ends."""
    if not token:
        print(f"Not saved to the Hub (no token): {message}")
        return None
    from huggingface_hub import HfApi

    destination = f"cord-runs/{run_id}"
    HfApi(token=token).upload_folder(
        repo_id=repo_id, folder_path=str(root), path_in_repo=destination,
        allow_patterns=list(patterns), ignore_patterns=["**/checkpoints/**", "*/checkpoints/*"],
        commit_message=f"DAPTA {run_id}: {message}",
    )
    print(f"Saved to https://huggingface.co/{repo_id}/tree/main/{destination} ({message})", flush=True)
    return destination


def write_json(path: Path, value: Any) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, default=str) + "\n", encoding="utf-8")
    return path

## Cell 5: settings
Edit here only. The extraction prompt is fixed in the engine so results stay comparable with the baseline.

In [ ]:
import importlib, json, math, random, sys, time
from pathlib import Path
import torch

sys.path.insert(0, "/content")
import dapta_vlm_engine as engine
importlib.reload(engine)

CFG = engine.EngineConfig(
    image_long_side=1280,   # same resize for training and evaluation
    max_seq_len=4096,       # every example is checked against this; nothing is silently truncated
    max_new_tokens=1536,    # matches the zero-shot baseline
).validate()
ROOT = Path(CFG.root); ROOT.mkdir(parents=True, exist_ok=True)

SMOKE_TRAIN, SMOKE_STEPS, SMOKE_EVAL = 12, 5, [20, 21, 22]   # eval receipts are outside the tuning subset
BATCH_CANDIDATES, EFFECTIVE_BATCH, BENCH_STEPS = [1, 2, 4], 8, 5
TUNE_TRAIN_SIZE, TUNE_VAL_INDICES, TUNE_STEPS = 256, list(range(20)), 35
TUNE_TRIALS = [{"rank": 8, "lr": 1e-4}, {"rank": 16, "lr": 1e-4}, {"rank": 16, "lr": 2e-4}]
FULL_EPOCHS, EVAL_EVERY, SAVE_EVERY, PATIENCE = 3, 25, 25, 2

engine.set_seed(CFG.seed)
torch.backends.cuda.matmul.allow_tf32 = True

# Results are copied to your private Hub repo after each stage: Colab deletes /content when a runtime ends.
import getpass
HF_REPO = "Pavan2020/DAPTA"
RUN_ID = time.strftime("%Y%m%d-%H%M%S")
HF_TOKEN = getpass.getpass("Hugging Face write token (press Enter to skip saving to the Hub): ").strip() or None
def persist(patterns, message):
    return engine.persist_to_hub(ROOT, HF_REPO, HF_TOKEN, RUN_ID, patterns, message)
print(json.dumps({"config": CFG.__dict__, "engine": engine.__file__}, indent=2))

## Cell 6: CORD data, schema and length checks
Pinned dataset revision. Conversations are built lazily, so images are decoded and resized on access.

In [ ]:
from datasets import load_dataset

train_raw = load_dataset(CFG.dataset_id, revision=CFG.dataset_revision, split="train")
val_raw = load_dataset(CFG.dataset_id, revision=CFG.dataset_revision, split="validation")
assert (len(train_raw), len(val_raw)) == (800, 100), (len(train_raw), len(val_raw))
print("schema:", engine.check_rows(train_raw, range(len(train_raw))), engine.check_rows(val_raw, range(len(val_raw))))

rng = random.Random(CFG.seed)
train_order = list(range(len(train_raw))); rng.shuffle(train_order)
smoke_ds = engine.receipt_dataset(train_raw, train_order[:SMOKE_TRAIN], CFG)
tune_train_ds = engine.receipt_dataset(train_raw, train_order[:TUNE_TRAIN_SIZE], CFG)
full_train_ds = engine.receipt_dataset(train_raw, range(len(train_raw)), CFG)
tune_val_ds = engine.receipt_dataset(val_raw, TUNE_VAL_INDICES, CFG)
smoke_val_ds = engine.receipt_dataset(val_raw, SMOKE_EVAL, CFG)
full_val_ds = engine.receipt_dataset(val_raw, range(len(val_raw)), CFG)
# Largest images and longest answers, so the benchmark measures worst-case memory.
stress_ids = sorted(train_order[:96], key=lambda i: train_raw[i]["image"].width * train_raw[i]["image"].height
                    + 128 * len(train_raw[i]["ground_truth"]), reverse=True)[:40]
stress_ds = engine.receipt_dataset(train_raw, stress_ids, CFG)

from transformers import AutoProcessor
_processor = AutoProcessor.from_pretrained(CFG.model_id)
LENGTHS = {
    "train": engine.length_report(engine.sequence_lengths(_processor, full_train_ds), CFG.max_seq_len),
    "validation": engine.length_report(engine.sequence_lengths(_processor, full_val_ds), CFG.max_seq_len),
}
del _processor
engine.write_json(ROOT / "data_report.json", {"lengths": LENGTHS, "stress_ids": stress_ids})
print(json.dumps(LENGTHS, indent=2))

## Cell 7: skill baseline (base model, short prompt vs whole skill)
The 4-bit base model, no adapter, on all 100 validation receipts, twice:
1. **Short prompt**: the baseline's prompt, CORD-format answers, graded by the baseline scorer.
2. **Whole skill**: the repo's `receipt-to-expense` skill document and output schema, unchanged, as the system prompt.

The skill asks for a different format (expense report, amounts as plain numbers), so both conditions are also graded on that
format with the skill's own `evaluation_plan.json` rules; CORD gold and the short-prompt answers are converted with the same
code. Amount parsing reads Indonesian `18.000` as eighteen thousand.

In [ ]:
import hashlib, jsonschema
RUN_SKILL_BASELINE = True

# The repo's receipt-to-expense skill, verbatim, checked against the reviewed contract's hashes.
SKILL_SPEC = json.loads(r'''{
  "name": "receipt_to_expense_report",
  "version": "1.0.0",
  "description": "Convert one or more receipt images into a normalized expense report containing only visible evidence.",
  "input": {
    "modality": "receipt_images",
    "cardinality": "one_or_more",
    "constraints": [
      "each image is processed as one independent receipt object",
      "no external knowledge, lookup, or context beyond the image pixels"
    ]
  },
  "output": "strict JSON validating against output_schema; receipts array length equals number of input images",
  "extraction_rules": [
    "Extract only text visibly printed on the receipt; never guess, impute, or complete partial values.",
    "Use null for any field not explicitly visible; never use empty strings, placeholders, or defaults.",
    "merchant: merchant/store name exactly as visible.",
    "address: address text transcribed as printed; null if no address is visible.",
    "transaction_date: visible transaction date normalized to ISO 8601 YYYY-MM-DD; null if absent or unresolvably ambiguous.",
    "currency: ISO 4217 alpha-3 code only when a currency code or unambiguous symbol is explicitly printed; otherwise null; never infer from locale, language, merchant, or address.",
    "subtotal, tax, service_charge, total: numeric values as printed; null when the corresponding labeled line is absent.",
    "line_items: one entry per visible purchased item; description as printed; quantity as printed number else null; amount is the printed line total."
  ],
  "normalization": {
    "amounts": "plain decimal numbers; strip currency symbols and thousands separators",
    "dates": "YYYY-MM-DD",
    "strings": "trim leading/trailing whitespace; collapse internal whitespace runs"
  },
  "exclusions": [
    "no policy approval determination",
    "no reimbursement eligibility determination",
    "no expense category or GL code assignment",
    "no currency inference",
    "no fields beyond output_schema",
    "no commentary, markdown, or chain-of-thought in output"
  ],
  "training_data": {
    "datasets": [
      "CORD v2 (pinned)",
      "SROIE (pinned)"
    ],
    "label_mapping": {
      "CORD_v2": {
        "menu": "line_items",
        "service": "service_charge",
        "sub_total": "subtotal",
        "tax": "tax",
        "total": "total"
      },
      "SROIE": {
        "address": "address",
        "company": "merchant",
        "date": "transaction_date",
        "total": "total"
      }
    }
  }
}''')
OUTPUT_SCHEMA = json.loads(r'''{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "receipts": {
      "type": "array",
      "minItems": 1,
      "items": {
        "type": "object",
        "additionalProperties": false,
        "properties": {
          "merchant": {
            "type": ["string", "null"]
          },
          "address": {
            "type": ["string", "null"]
          },
          "transaction_date": {
            "type": ["string", "null"],
            "pattern": "^\\d{4}-\\d{2}-\\d{2}$"
          },
          "currency": {
            "type": ["string", "null"],
            "pattern": "^[A-Z]{3}$"
          },
          "subtotal": {
            "type": ["number", "null"]
          },
          "tax": {
            "type": ["number", "null"]
          },
          "service_charge": {
            "type": ["number", "null"]
          },
          "total": {
            "type": ["number", "null"]
          },
          "line_items": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": false,
              "properties": {
                "description": {
                  "type": ["string", "null"]
                },
                "quantity": {
                  "type": ["number", "null"]
                },
                "amount": {
                  "type": ["number", "null"]
                }
              },
              "required": ["description", "quantity", "amount"]
            }
          }
        },
        "required": [
          "merchant",
          "address",
          "transaction_date",
          "currency",
          "subtotal",
          "tax",
          "service_charge",
          "total",
          "line_items"
        ]
      }
    }
  },
  "required": ["receipts"]
}''')
REVIEWED_SHA256 = {"skill_spec.json": "53e2752024304c966926f386974a6207d441f21b6de9aec2ee28dd2bf90ee89b", "output_schema.json": "8fbecf6888ae3320d5a17e548f9c63aeb7ca1c7d0af049dc6f1fee43c43d3823", "evaluation_plan.json": "81802ef4df48f953ddcbf8d946a9b88f8857bb049cd03e1b237dc2c4279dcc13"}
for name, value in (("skill_spec.json", SKILL_SPEC), ("output_schema.json", OUTPUT_SCHEMA)):
    digest = hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
    assert digest == REVIEWED_SHA256[name], f"{name} differs from the reviewed skill"
FULL_SKILL = engine.full_skill_prompt(SKILL_SPEC, OUTPUT_SCHEMA)

if not RUN_SKILL_BASELINE:
    print("Skipped: skill baseline.")
else:
    base_model, base_processor = engine.load_base(CFG)
    everything = range(len(val_raw))
    short = engine.evaluate_condition(base_model, base_processor, val_raw, everything, CFG, desc="base + short prompt",
                                      user_prompt=CFG.prompt, answer_format="cord", output_schema=OUTPUT_SCHEMA,
                                      output_dir=ROOT / "baseline_4bit_short_prompt")["summary"]
    skill = engine.evaluate_condition(base_model, base_processor, val_raw, everything, CFG, desc="base + full skill",
                                      system_prompt=FULL_SKILL, user_prompt=engine.SKILL_USER_PROMPT,
                                      answer_format="expense", output_schema=OUTPUT_SCHEMA,
                                      output_dir=ROOT / "baseline_4bit_full_skill")["summary"]
    del base_model, base_processor
    engine.free_gpu()
    same = short["expense_format"]
    BASELINES = {"base_short_prompt": short, "base_full_skill": skill, "bf16_short_prompt_reference": engine.BASELINE_BF16}
    engine.write_json(ROOT / "skill_baseline.json", BASELINES)

    def cell(value, digits=3):
        return "n/a" if value is None else (f"{value:.{digits}f}" if isinstance(value, float) else str(value))
    table = [
        ("Skill-format field F1", same["field_f1"], skill["field_f1"]),
        ("Skill-format field precision", same["field_precision"], skill["field_precision"]),
        ("Skill-format field recall", same["field_recall"], skill["field_recall"]),
        ("Line-item F1", same["line_item_f1"], skill["line_item_f1"]),
        ("Total accuracy", same["total_accuracy"], skill["total_accuracy"]),
        ("Subtotal accuracy", same["subtotal_accuracy"], skill["subtotal_accuracy"]),
        ("Tax accuracy", same["tax_accuracy"], skill["tax_accuracy"]),
        ("Valid JSON", short["valid_json_fraction"], skill["valid_json_fraction"]),
        ("Schema valid (skill gate)", None, skill["schema_valid_fraction"]),
        ("CORD field F1 (baseline scorer)", short["field_f1"], None),
        ("Mean input tokens", short["mean_input_tokens"], skill["mean_input_tokens"]),
        ("Mean output tokens", short["mean_output_tokens"], skill["mean_output_tokens"]),
        ("Seconds per receipt", short["mean_seconds_per_receipt"], skill["mean_seconds_per_receipt"]),
        ("Truncated outputs", short["truncated_outputs"], skill["truncated_outputs"]),
    ]
    print("BASELINE RESULTS (4-bit base, 100 validation receipts)")
    print(f"{'Metric':34s} | {'base + short prompt':>20s} | {'base + full skill':>18s}")
    for name, a, b in table:
        print(f"BASELINE ROW {name:34s} | {cell(a):>20s} | {cell(b):>18s}")
    print("Reference, BF16 + short prompt (earlier run, CORD scorer): field F1", engine.BASELINE_BF16["field_f1"])
    persist(["skill_baseline.json", "data_report.json", "baseline_4bit_*/**"], "skill baseline")
    print("SKILL BASELINE DONE")

## Cell 8: GPU compatibility tests
Real 4-bit model, Unsloth collator, LoRA. Raises on the first failure.

In [ ]:
model, processor = engine.load_lora(CFG, rank=8)
collator = engine.make_collator(model, processor, CFG)
TESTS = {"parameters": engine.trainable_report(model)}
TESTS["collator_batch_of_2"] = engine.probe_collator(collator, processor, smoke_ds, rows=2)
TESTS["gradients"] = engine.gradient_check(model, collator, smoke_ds)
TESTS["save_reload"] = engine.save_reload_check(model, processor, CFG, ROOT / "tests" / "adapter")
del model, processor, collator
TESTS["memory_after_release"] = engine.free_gpu()
engine.write_json(ROOT / "tests.json", TESTS)
print(json.dumps(TESTS, indent=2))
print("GPU TESTS PASSED")

## Cell 9: smoke test
12 receipts, 5 optimizer steps, rank 8, lr 1e-4, batch 1 x accumulation 2. Then generation on 3 separate validation receipts.

In [ ]:
SMOKE_PASSED = False
spec = engine.TrainSpec("smoke", rank=8, lr=1e-4, batch=1, accum=2, steps=SMOKE_STEPS)
smoke, model, processor, trainer = engine.train_run(spec, CFG, smoke_ds, val_ds=smoke_val_ds)
assert smoke["stable"], f"non-finite training loss: {smoke}"
assert smoke["steps_done"] == SMOKE_STEPS
generation = engine.evaluate_receipts(model, processor, val_raw, SMOKE_EVAL, CFG,
                                      desc="smoke eval", output_dir=ROOT / "smoke" / "generation", print_every=1)
smoke["generation"] = generation["summary"]
del model, processor, trainer
smoke["memory_after_release"] = engine.free_gpu()
engine.write_json(ROOT / "smoke_metrics.json", smoke)
print(json.dumps(smoke, indent=2))
SMOKE_PASSED = True
print("SMOKE TEST PASSED")

## Cell 10: microbatch benchmark
Real training steps on the 40 largest receipts, effective batch 8. An OOM is recorded, not hidden.

In [ ]:
assert SMOKE_PASSED, "run the smoke test first"
BENCH = []
steps_per_epoch = math.ceil(len(full_train_ds) / EFFECTIVE_BATCH)
for batch in BATCH_CANDIDATES:
    accum = EFFECTIVE_BATCH // batch
    print(f"\n=== microbatch {batch} x accumulation {accum} ===", flush=True)
    row = {"microbatch": batch, "grad_accum": accum, "oom": False}
    try:
        result, model, processor, trainer = engine.train_run(
            engine.TrainSpec(f"bench_bs{batch}", rank=16, lr=1e-4, batch=batch, accum=accum, steps=BENCH_STEPS),
            CFG, stress_ds)
        row.update({k: result[k] for k in ("stable", "median_step_seconds", "examples_per_second",
                                            "peak_allocated_gib", "peak_reserved_gib", "total_gib",
                                            "gpu_utilization_mean", "first_loss", "last_loss")})
        row["estimated_full_training_minutes"] = round(steps_per_epoch * FULL_EPOCHS * result["median_step_seconds"] / 60, 1)
        del model, processor, trainer
    except torch.OutOfMemoryError as error:
        row.update({"oom": True, "error": str(error).splitlines()[0]})
    engine.free_gpu()
    BENCH.append(row)
    print(json.dumps(row, indent=2), flush=True)

capacity = torch.cuda.get_device_properties(0).total_memory / 2**30
CHOICE = engine.choose_batch(BENCH, capacity_gib=capacity)
BEST_BATCH, BEST_ACCUM = CHOICE["microbatch"], CHOICE["grad_accum"]
engine.write_json(ROOT / "batch_benchmark.json", {"rows": BENCH, "choice": CHOICE, "capacity_gib": round(capacity, 2)})
print("CHOICE:", json.dumps(CHOICE, indent=2))

## Cell 11: hyperparameter trials
Each trial loads fresh base weights and a new adapter. Selection: validation field F1, then validation loss.

In [ ]:
HPO, PREDICTIONS = [], {}
for number, trial in enumerate(TUNE_TRIALS, 1):
    name = f"hpo_{number}_r{trial['rank']}_lr{trial['lr']:g}"
    print(f"\n===== trial {number}/{len(TUNE_TRIALS)}: {name} =====", flush=True)
    spec = engine.TrainSpec(name, rank=trial["rank"], lr=trial["lr"], batch=BEST_BATCH, accum=BEST_ACCUM,
                            steps=TUNE_STEPS, log_every=5)
    result, model, processor, trainer = engine.train_run(spec, CFG, tune_train_ds, val_ds=tune_val_ds)
    evaluation = engine.evaluate_receipts(model, processor, val_raw, TUNE_VAL_INDICES, CFG,
                                          desc=name, output_dir=ROOT / name / "generation", print_every=5)
    del model, processor, trainer
    engine.free_gpu()
    summary = evaluation["summary"]
    result.update({"val_field_f1": summary["field_f1"], "val_field_precision": summary["field_precision"],
                   "val_field_recall": summary["field_recall"], "val_valid_json": summary["valid_json_fraction"],
                   "val_total_price_exact": summary["total_price_exact_fraction"]})
    HPO.append(result)
    PREDICTIONS[name] = evaluation["records"]
    engine.write_json(ROOT / "hpo_results.json", HPO)
    print(json.dumps({k: result[k] for k in ("name", "rank", "lr", "training_loss", "val_loss", "val_field_f1",
                                             "val_field_precision", "val_field_recall", "gpu_utilization_mean")}, indent=2), flush=True)

DECISION = engine.choose_trial(HPO, PREDICTIONS)
engine.write_json(ROOT / "best_hpo.json", DECISION)
print(json.dumps({k: v for k, v in DECISION.items() if k != "best"}, indent=2))

## Cell 12: recommendation and approval gate
Full training does **not** start until you set `APPROVE_FULL_TRAINING = True` and run cells 13-16.

In [ ]:
# Tuning results come from this runtime, else its saved files, else the approved outcome recorded on 2026-10-17
# (the runtime that ran tuning ended: hpo_2 r16 lr1e-4 F1 0.808 on 20 validation receipts, microbatch 4 x 2).
if "DECISION" not in globals() and (ROOT / "best_hpo.json").exists():
    DECISION = json.loads((ROOT / "best_hpo.json").read_text())
    _bench = json.loads((ROOT / "batch_benchmark.json").read_text())
    BENCH, BEST_BATCH, BEST_ACCUM = _bench["rows"], _bench["choice"]["microbatch"], _bench["choice"]["grad_accum"]
if "DECISION" not in globals():
    DECISION = {"best": {"name": "hpo_2_r16_lr0.0001", "rank": 16, "lr": 1e-4, "val_field_f1": 0.80798},
                "conclusive": False, "source": "approved outcome; tuning files lost with the earlier runtime",
                "recommendation": "hpo_2 (lr 1e-4) and hpo_3 (lr 2e-4) were not separable on 20 validation receipts."}
    BENCH = [{"microbatch": 4, "grad_accum": 2, "estimated_full_training_minutes": 26.4}]
    BEST_BATCH, BEST_ACCUM = 4, 2
best = DECISION["best"]
RECOMMENDED = engine.TrainSpec("full", rank=int(best["rank"]), lr=float(best["lr"]), batch=BEST_BATCH, accum=BEST_ACCUM,
                               epochs=FULL_EPOCHS, eval_every=EVAL_EVERY, save_every=SAVE_EVERY,
                               early_stop_patience=PATIENCE, log_every=5).validate()
bench = next(r for r in BENCH if r["microbatch"] == BEST_BATCH)
print(json.dumps({"recommended": RECOMMENDED.__dict__, "conclusive": DECISION["conclusive"],
                  "note": DECISION.get("recommendation", "best trial is separable from the runner-up"),
                  "total_steps": RECOMMENDED.total_steps(len(full_train_ds)),
                  "estimated_training_minutes_excluding_eval": bench.get("estimated_full_training_minutes")}, indent=2))
# Approved by Pavan on 2026-10-17 after reviewing tuning: rank 16, lr 1e-4, batch 4 x 2, 3 epochs.
APPROVE_FULL_TRAINING = True
assert (RECOMMENDED.rank, RECOMMENDED.lr, RECOMMENDED.batch, RECOMMENDED.accum) == (16, 1e-4, 4, 2), RECOMMENDED
print("Full training approved: run cell 13.")

## Cell 13: full training (800 receipts)
Checkpoints every `SAVE_EVERY` steps, early stopping on validation loss, best checkpoint restored. If Colab disconnects or runs out of memory, rerun this cell: it resumes from the last checkpoint.

In [ ]:
FULL_DONE = False
if not globals().get("APPROVE_FULL_TRAINING"):
    print("Skipped: full training not approved.")
else:
    try:
        FULL, full_model, full_processor, full_trainer = engine.train_run(
            RECOMMENDED, CFG, full_train_ds, val_ds=full_val_ds, save=True, resume=True)
    except torch.OutOfMemoryError as error:
        print("OUT OF MEMORY:", str(error).splitlines()[0])
        print("Checkpoints are kept. Lower the microbatch (keep the effective batch), then rerun this cell to resume.")
        raise
    engine.write_json(ROOT / "full_summary.json", FULL)
    print(json.dumps(FULL, indent=2))
    persist(["full_summary.json", "full/adapter/**", "full/train_summary.json", "full/train_history.jsonl"], "trained adapter")
    FULL_DONE = True

## Cell 14: final comparison (100 validation receipts)
All conditions use the same 4-bit base weights, image preprocessing, greedy decoding and receipts. The two base-model
conditions reuse cell 7's saved predictions; only the LoRA is generated here.

In [ ]:
if not globals().get("FULL_DONE"):
    print("Skipped: needs a finished full training run.")
else:
    del full_trainer
    engine.free_gpu()
    everything = range(len(val_raw))
    lora = engine.evaluate_condition(full_model, full_processor, val_raw, everything, CFG, desc="LoRA + short prompt",
                                     user_prompt=CFG.prompt, answer_format="cord", output_schema=OUTPUT_SCHEMA,
                                     output_dir=ROOT / "full" / "validation")["summary"]
    base_model, base_processor = engine.load_base(CFG)
    base = engine.evaluate_condition(base_model, base_processor, val_raw, everything, CFG, desc="base + short prompt",
                                     user_prompt=CFG.prompt, answer_format="cord", output_schema=OUTPUT_SCHEMA,
                                     output_dir=ROOT / "baseline_4bit_short_prompt")["summary"]
    skill = engine.evaluate_condition(base_model, base_processor, val_raw, everything, CFG, desc="base + full skill",
                                      system_prompt=FULL_SKILL, user_prompt=engine.SKILL_USER_PROMPT,
                                      answer_format="expense", output_schema=OUTPUT_SCHEMA,
                                      output_dir=ROOT / "baseline_4bit_full_skill")["summary"]
    del base_model, base_processor
    engine.free_gpu()

    b, k, l = base, skill, lora
    be, le = base["expense_format"], lora["expense_format"]
    rows = [
        ("CORD field F1 (baseline scorer)", b["field_f1"], None, l["field_f1"]),
        ("CORD field precision", b["field_precision"], None, l["field_precision"]),
        ("CORD field recall", b["field_recall"], None, l["field_recall"]),
        ("CORD total price exact", b["total_price_exact_fraction"], None, l["total_price_exact_fraction"]),
        ("CORD document exact", b["document_exact_fraction"], None, l["document_exact_fraction"]),
        ("Skill-format field F1", be["field_f1"], k["field_f1"], le["field_f1"]),
        ("Skill-format line-item F1", be["line_item_f1"], k["line_item_f1"], le["line_item_f1"]),
        ("Skill-format total accuracy", be["total_accuracy"], k["total_accuracy"], le["total_accuracy"]),
        ("Skill-format subtotal accuracy", be["subtotal_accuracy"], k["subtotal_accuracy"], le["subtotal_accuracy"]),
        ("Valid JSON", b["valid_json_fraction"], k["valid_json_fraction"], l["valid_json_fraction"]),
        ("Schema valid (skill gate)", None, k["schema_valid_fraction"], None),
        ("Mean input tokens", b["mean_input_tokens"], k["mean_input_tokens"], l["mean_input_tokens"]),
        ("Mean output tokens", b["mean_output_tokens"], k["mean_output_tokens"], l["mean_output_tokens"]),
        ("Seconds per receipt", b["mean_seconds_per_receipt"], k["mean_seconds_per_receipt"], l["mean_seconds_per_receipt"]),
        ("Truncated outputs", b["truncated_outputs"], k["truncated_outputs"], l["truncated_outputs"]),
        ("Peak GPU memory (GiB)", b["peak_allocated_gib"], k["peak_allocated_gib"], l["peak_allocated_gib"]),
    ]
    def cell(value):
        return "n/a" if value is None else (f"{value:.3f}" if isinstance(value, float) else str(value))
    print("FINAL COMPARISON (4-bit Qwen3-VL-8B, 100 CORD validation receipts)")
    print(f"FINAL ROW {'Metric':32s} | {'base':>10s} | {'base + full skill':>17s} | {'LoRA':>10s}")
    for name, x, y, z in rows:
        print(f"FINAL ROW {name:32s} | {cell(x):>10s} | {cell(y):>17s} | {cell(z):>10s}")
    print("FINAL ROW Base + cached skill: not measured (needs a verified prefix-cache decoder; accuracy is identical to the full skill by construction)")
    print(f"FINAL ROW Reference: BF16 base + short prompt, earlier run: CORD field F1 {engine.BASELINE_BF16['field_f1']}")
    FINAL_COMPARISON_REPORT = {
        "base_short_prompt": base, "base_full_skill": skill, "lora_short_prompt": lora,
        "base_cached_skill": "not measured", "bf16_reference": engine.BASELINE_BF16,
        "lora_vs_base_cord": engine.compare(lora, base),
        "notes": ["20 of the 100 validation receipts were used for hyperparameter selection.",
                  "Skill-format metrics follow research/adapter-assets/skills/receipt-to-expense/evaluation_plan.json.",
                  "Latency uses an unmerged 4-bit LoRA through Unsloth generate; not a serving benchmark."],
    }
    engine.write_json(ROOT / "final_comparison.json", FINAL_COMPARISON_REPORT)
    print("FINAL COMPARISON DONE")

## Cell 15: final held-out test (run once, after every decision is made)

In [ ]:
RUN_FINAL_TEST = True  # selection is final (approved configuration); the test split is scored once
if not (RUN_FINAL_TEST and globals().get("FULL_DONE")):
    print("Test split untouched. Set RUN_FINAL_TEST = True only when the configuration is final.")
else:
    test_raw = load_dataset(CFG.dataset_id, revision=CFG.dataset_revision, split="test")
    assert len(test_raw) == 100
    FINAL = engine.evaluate_condition(full_model, full_processor, test_raw, range(len(test_raw)), CFG,
                                      desc="FINAL test, LoRA", user_prompt=CFG.prompt, answer_format="cord",
                                      output_schema=OUTPUT_SCHEMA, output_dir=ROOT / "full" / "final_test")["summary"]
    engine.write_json(ROOT / "final_test.json", FINAL)
    print(json.dumps(FINAL, indent=2))

## Cell 16: export
Collects metrics and the adapter into one zip. `EXPORT_TO_HF = True` also uploads it to your private Hugging Face repo.

In [ ]:
import shutil
EXPORT_TO_HF, HF_REPO = False, "Pavan2020/DAPTA"
reports = sorted(p.relative_to(ROOT).as_posix() for p in ROOT.rglob("*.json") if "checkpoints" not in p.parts)
print("Reports:", json.dumps(reports, indent=2))
staging = Path("/content/dapta_export"); shutil.rmtree(staging, ignore_errors=True)
for rel in reports:
    (staging / rel).parent.mkdir(parents=True, exist_ok=True); shutil.copy2(ROOT / rel, staging / rel)
if (ROOT / "full" / "adapter").is_dir():
    shutil.copytree(ROOT / "full" / "adapter", staging / "full" / "adapter")
archive = shutil.make_archive("/content/dapta_cord_lora_export", "zip", root_dir=staging)
print("Archive:", archive)
if EXPORT_TO_HF:
    import getpass
    from huggingface_hub import HfApi
    run_id = time.strftime("%Y%m%d-%H%M%S")
    HfApi(token=getpass.getpass("Hugging Face write token: ")).upload_folder(
        repo_id=HF_REPO, folder_path=str(staging), path_in_repo=f"cord-runs/{run_id}",
        commit_message=f"DAPTA CORD run {run_id}")
    print("Uploaded to", f"{HF_REPO}/cord-runs/{run_id}")

## Cell 17: vLLM server environment
The four-way benchmark serves every condition through vLLM. vLLM pins its own torch, so it gets a separate virtual
environment; the training packages in this kernel are not touched. Takes about 5 minutes once.

**Before cells 17-20, restart the kernel** (Kernel > Restart): it frees the GPU for vLLM, and every file in `/content` is kept.

In [ ]:
# Separate environment for the vLLM server: vLLM pins torch 2.13 / transformers 5, the training stack uses
# torch 2.11 / transformers 4.57.6. Nothing here changes the notebook kernel's packages.
import os, subprocess, sys
VLLM_ENV = "/content/vllm-env"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.12.19"], check=True)
if not os.path.exists(f"{VLLM_ENV}/bin/python"):
    subprocess.run(["uv", "venv", VLLM_ENV, "--python", sys.executable], check=True)
subprocess.run(["uv", "pip", "install", "--python", f"{VLLM_ENV}/bin/python", "-q",
                "vllm==0.30.0", "transformers==5.17.0"], check=True)
check = subprocess.run([f"{VLLM_ENV}/bin/python", "-c",
                        "import vllm, torch, transformers; print('vllm', vllm.__version__, '| torch', torch.__version__, "
                        "'| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())"],
                       capture_output=True, text=True)
print(check.stdout, check.stderr[-2000:])
assert check.returncode == 0 and "vllm 0.30.0" in check.stdout, "vLLM environment check failed"
print("VLLM ENV OK")

## Cell 18: merged adapter (fallback only)
Run with `MERGE_ADAPTER = True` only if vLLM rejects the adapter in the quick run; then set `LORA_MODE = "merged"` in cell 20.

In [ ]:
# Only needed when vLLM cannot load the adapter directly (LORA_MODE = "merged" in cell 20).
MERGE_ADAPTER = False
ADAPTER, MERGED = "/content/dapta_lora/full/adapter", "/content/dapta_lora/full/merged_bf16"
if not MERGE_ADAPTER:
    print("Skipped: adapter is served directly by vLLM (--enable-lora).")
else:
    import subprocess, sys, textwrap
    script = textwrap.dedent(f'''
        import torch
        from peft import PeftModel
        from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
        base = Qwen3VLForConditionalGeneration.from_pretrained("Qwen/Qwen3-VL-8B-Instruct", dtype=torch.bfloat16, device_map="cpu")
        merged = PeftModel.from_pretrained(base, "{ADAPTER}").merge_and_unload()
        merged.save_pretrained("{MERGED}", safe_serialization=True)
        AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct").save_pretrained("{MERGED}")
        print("merged into", "{MERGED}")
    ''')
    # A subprocess keeps the 16 GB merge out of this kernel's memory.
    subprocess.run([sys.executable, "-c", script], check=True)

## Cell 19: benchmark module
Written verbatim from `notebooks/dapta_bench.py` (sha `7e7157ac15f9`).

In [ ]:
%%writefile /content/dapta_bench.py
"""Four-way serving benchmark through vLLM: base vs full skill (uncached, cached) vs LoRA.

The notebook kernel runs this client; vLLM runs as a separate server process from its own
virtual environment (it pins a different torch than the Unsloth training stack). Quality,
cost, speed and scale come from the same streamed requests. Nothing here imports torch.
"""

from __future__ import annotations

import asyncio
import base64
import concurrent.futures
import csv
import io
import json
import math
import os
import random
import re
import statistics
import subprocess
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Mapping, Sequence

import dapta_vlm_engine as engine

CONDITIONS = ("base", "skill_uncached", "skill_cached", "lora")
# Same weights and settings everywhere; only prefix caching and the adapter differ between servers.
SERVERS = {
    "plain": {"conditions": ("base", "skill_uncached"), "prefix_cache": False, "lora": False},
    "cached": {"conditions": ("skill_cached",), "prefix_cache": True, "lora": False},
    "lora": {"conditions": ("lora",), "prefix_cache": False, "lora": True},
}
SKILL_CONDITIONS = ("skill_uncached", "skill_cached")


@dataclass(frozen=True)
class BenchConfig:
    model_id: str = "Qwen/Qwen3-VL-8B-Instruct"
    dtype: str = "bfloat16"
    adapter_dir: str = "/content/dapta_lora/full/adapter"
    lora_mode: str = "adapter"  # "adapter": vLLM --enable-lora; "merged": adapter merged into BF16 weights
    merged_dir: str = "/content/dapta_lora/full/merged_bf16"
    vllm_python: str = "/content/vllm-env/bin/python"
    host: str = "127.0.0.1"
    port: int = 8000
    max_model_len: int = 8192
    gpu_memory_utilization: float = 0.85
    max_new_tokens: int = 1536
    concurrencies: tuple[int, ...] = (1, 8, 32)
    repeats: int = 3
    warmup_requests: int = 3
    quick_docs: int = 5
    bootstrap_samples: int = 2000
    seed: int = 42
    gpu_hour_usd: float = 1.50
    gpu_hour_note: str = "Assumed on-demand price of one A100 40GB GPU-hour; edit to your actual rate."
    results_dir: str = "/content/dapta_lora/results"
    server_start_timeout_s: int = 1800
    identical_output_sample: int = 10
    server_command_override: tuple[str, ...] | None = None  # tests only: replaces the vLLM command

    def validate(self) -> "BenchConfig":
        _require(self.lora_mode in ("adapter", "merged"), "lora_mode must be 'adapter' or 'merged'")
        _require(self.repeats >= 3, "timing needs at least 3 repeats")
        _require(all(c >= 1 for c in self.concurrencies), "concurrency must be >= 1")
        _require(1 in self.concurrencies, "concurrency 1 is required for latency and quality")
        _require(self.gpu_hour_usd > 0, "gpu_hour_usd must be positive")
        _require(0.3 <= self.gpu_memory_utilization <= 0.95, "gpu_memory_utilization must be 0.3..0.95")
        return self


def _require(condition: bool, message: str) -> None:
    if not condition:
        raise ValueError(message)


# --------------------------------------------------------------------------- documents and prompts


def encode_docs(split: Any, indices: Sequence[int], cfg: engine.EngineConfig) -> list[dict[str, Any]]:
    """Each receipt resized exactly as in training and evaluation, sent as a lossless PNG."""
    docs = []
    for index in indices:
        row = split[int(index)]
        image = engine.prepare_image(row["image"], cfg.image_long_side)
        buffer = io.BytesIO()
        image.save(buffer, format="PNG")
        docs.append({"doc_id": int(index), "gold": engine.gold_parse(row),
                     "data_url": "data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode()})
    return docs


def messages_for(condition: str, doc: Mapping[str, Any], full_skill: str) -> list[dict[str, Any]]:
    image = {"type": "image_url", "image_url": {"url": doc["data_url"]}}
    if condition in ("base", "lora"):
        return [{"role": "user", "content": [image, {"type": "text", "text": engine.PROMPT}]}]
    # The skill is the system message, so it is an identical prefix before every image.
    return [{"role": "system", "content": full_skill},
            {"role": "user", "content": [image, {"type": "text", "text": engine.SKILL_USER_PROMPT}]}]


def served_model(condition: str) -> str:
    return "lora" if condition == "lora" else "base"


# --------------------------------------------------------------------------- server lifecycle


def adapter_rank(adapter_dir: str) -> int:
    config = json.loads((Path(adapter_dir) / "adapter_config.json").read_text())
    return int(config["r"])


def server_command(cfg: BenchConfig, server: str) -> list[str]:
    spec = SERVERS[server]
    if cfg.server_command_override:
        return [*cfg.server_command_override, "--port", str(cfg.port),
                "--prefix-cache", "on" if spec["prefix_cache"] else "off"]
    merged = spec["lora"] and cfg.lora_mode == "merged"
    command = [
        cfg.vllm_python, "-m", "vllm.entrypoints.openai.api_server",
        "--model", cfg.merged_dir if merged else cfg.model_id,
        "--served-model-name", "lora" if merged else "base",
        "--dtype", cfg.dtype,
        "--max-model-len", str(cfg.max_model_len),
        "--gpu-memory-utilization", str(cfg.gpu_memory_utilization),
        "--seed", str(cfg.seed),
        "--generation-config", "vllm",  # ignore the model's sampling defaults; requests set greedy decoding
        "--limit-mm-per-prompt", json.dumps({"image": 1, "video": 0}),
        "--mm-processor-cache-gb", "0",  # no cross-request image cache: repeats must not get cheaper
        "--host", cfg.host, "--port", str(cfg.port),
        "--enable-prefix-caching" if spec["prefix_cache"] else "--no-enable-prefix-caching",
    ]
    if spec["lora"] and not merged:
        command += ["--enable-lora", "--lora-modules", f"lora={cfg.adapter_dir}",
                    "--max-lora-rank", str(adapter_rank(cfg.adapter_dir)), "--max-loras", "1"]
    return command


class Server:
    """A vLLM server process with its log on disk; always stopped, even when a run fails."""

    def __init__(self, cfg: BenchConfig, server: str, log_dir: Path):
        self.cfg, self.server = cfg, server
        self.log_path = log_dir / f"server_{server}.log"
        self.base_url = f"http://{cfg.host}:{cfg.port}"
        self.process: subprocess.Popen | None = None
        self.log = None

    def __enter__(self) -> "Server":
        import httpx

        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        command = server_command(self.cfg, self.server)
        env = {**os.environ, "VLLM_SERVER_DEV_MODE": "1"}  # enables /reset_prefix_cache
        self.log = self.log_path.open("w", encoding="utf-8")
        self.log.write("$ " + " ".join(command) + "\n")
        self.log.flush()
        self.process = subprocess.Popen(command, stdout=self.log, stderr=subprocess.STDOUT, env=env)
        deadline = time.monotonic() + self.cfg.server_start_timeout_s
        started = time.monotonic()
        while time.monotonic() < deadline:
            if self.process.poll() is not None:
                raise RuntimeError(f"{self.server} server exited ({self.process.returncode}):\n{self.log_tail()}")
            try:
                if httpx.get(f"{self.base_url}/health", timeout=2).status_code == 200:
                    self.start_seconds = round(time.monotonic() - started, 1)
                    print(f"[server {self.server}] ready in {self.start_seconds}s", flush=True)
                    return self
            except httpx.HTTPError:
                pass
            time.sleep(2)
        self.__exit__(None, None, None)
        raise TimeoutError(f"{self.server} server not ready after {self.cfg.server_start_timeout_s}s:\n{self.log_tail()}")

    def __exit__(self, *exc: Any) -> None:
        if self.process and self.process.poll() is None:
            self.process.terminate()
            try:
                self.process.wait(timeout=90)
            except subprocess.TimeoutExpired:
                self.process.kill()
                self.process.wait(timeout=30)
        if self.log:
            self.log.close()
        # The next server needs the memory back; vLLM workers can release it a little after exit.
        deadline = time.monotonic() + 120
        while time.monotonic() < deadline:
            used = gpu_memory_used_mib()
            if used is None or used < 2500:
                return
            time.sleep(2)
        raise RuntimeError(f"GPU memory still in use after stopping the {self.server} server: {gpu_memory_used_mib()} MiB")

    def log_tail(self, lines: int = 40) -> str:
        try:
            return "\n".join(self.log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])
        except OSError:
            return "(no log)"


def gpu_memory_used_mib() -> int | None:
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10, check=True).stdout
        return int(out.strip().splitlines()[0])
    except (OSError, subprocess.SubprocessError, ValueError, IndexError):
        return None  # no nvidia-smi (tests): reported as unavailable


# --------------------------------------------------------------------------- client


async def stream_request(client: Any, base_url: str, condition: str, messages: list[dict[str, Any]],
                         cfg: BenchConfig) -> dict[str, Any]:
    """One streamed chat completion: TTFT from the first content token, usage from the final chunk."""
    payload = {
        "model": served_model(condition), "messages": messages, "stream": True,
        "stream_options": {"include_usage": True}, "temperature": 0.0, "max_tokens": cfg.max_new_tokens,
        "seed": cfg.seed, "chat_template_kwargs": {"enable_thinking": False},
    }
    started = time.perf_counter()
    first = None
    parts: list[str] = []
    usage = None
    async with client.stream("POST", f"{base_url}/v1/chat/completions", json=payload, timeout=None) as response:
        if response.status_code != 200:
            body = (await response.aread()).decode(errors="replace")[:500]
            raise RuntimeError(f"{condition}: HTTP {response.status_code}: {body}")
        async for line in response.aiter_lines():
            if not line.startswith("data:"):
                continue
            data = line[5:].strip()
            if data == "[DONE]":
                break
            chunk = json.loads(data)
            if chunk.get("usage"):
                usage = chunk["usage"]
            for choice in chunk.get("choices", []):
                piece = (choice.get("delta") or {}).get("content")
                if piece:
                    if first is None:
                        first = time.perf_counter()
                    parts.append(piece)
    ended = time.perf_counter()
    _require(usage is not None, f"{condition}: server returned no usage; token counts unavailable")
    completion = int(usage["completion_tokens"])
    ttft = (first or ended) - started
    latency = ended - started
    return {
        "text": "".join(parts), "ttft_s": ttft, "latency_s": latency,
        "prompt_tokens": int(usage["prompt_tokens"]), "completion_tokens": completion,
        "decode_tokens_per_s": (completion - 1) / (latency - ttft) if completion > 1 and latency > ttft else None,
    }


async def _metrics(client: Any, base_url: str) -> dict[str, float]:
    text = (await client.get(f"{base_url}/metrics", timeout=10)).text
    values: dict[str, float] = {}
    for line in text.splitlines():
        if line.startswith("#"):
            continue
        match = re.match(r"^(vllm:[a-z_]+)(?:\{[^}]*\})?\s+([-0-9.eE+]+)$", line.strip())
        if match:
            values[match.group(1)] = values.get(match.group(1), 0.0) + float(match.group(2))
    return values


def _kv_usage(values: Mapping[str, float]) -> float | None:
    for key in ("vllm:kv_cache_usage_perc", "vllm:gpu_cache_usage_perc"):
        if key in values:
            return values[key]
    return None


def _prefix_counters(values: Mapping[str, float]) -> tuple[float, float]:
    queries = next((v for k, v in values.items() if k.startswith("vllm:prefix_cache_queries")), 0.0)
    hits = next((v for k, v in values.items() if k.startswith("vllm:prefix_cache_hits")), 0.0)
    return queries, hits


async def timed_run(cfg: BenchConfig, base_url: str, condition: str, docs: Sequence[Mapping[str, Any]],
                    full_skill: str, concurrency: int) -> dict[str, Any]:
    """All docs at a fixed concurrency, with KV-cache usage sampled while requests are in flight."""
    import httpx

    limits = httpx.Limits(max_connections=max(64, concurrency * 2), max_keepalive_connections=64)
    async with httpx.AsyncClient(limits=limits) as client:
        before = await _metrics(client, base_url)
        gate = asyncio.Semaphore(concurrency)
        kv_peak: list[float] = []
        done = asyncio.Event()

        async def sample() -> None:
            while not done.is_set():
                usage = _kv_usage(await _metrics(client, base_url))
                if usage is not None:
                    kv_peak.append(usage)
                await asyncio.sleep(0.5)

        async def one(doc: Mapping[str, Any]) -> dict[str, Any]:
            async with gate:
                result = await stream_request(client, base_url, condition, messages_for(condition, doc, full_skill), cfg)
                return {"doc_id": doc["doc_id"], **result}

        sampler = asyncio.create_task(sample())
        started = time.perf_counter()
        try:
            results = await asyncio.gather(*(one(doc) for doc in docs))
        finally:
            done.set()
            await sampler
        wall = time.perf_counter() - started
        after = await _metrics(client, base_url)
    queries = _prefix_counters(after)[0] - _prefix_counters(before)[0]
    hits = _prefix_counters(after)[1] - _prefix_counters(before)[1]
    return {"results": results, "wall_s": wall, "kv_cache_usage_peak": max(kv_peak) if kv_peak else None,
            "prefix_cache_hit_rate": hits / queries if queries else None,
            "gpu_memory_used_mib": gpu_memory_used_mib()}


async def warm(cfg: BenchConfig, base_url: str, condition: str, warm_docs: Sequence[Mapping[str, Any]],
               full_skill: str, reset_prefix_cache: bool) -> None:
    """Warmup on validation receipts. The cached server is reset first, so only the skill prefix is warm."""
    import httpx

    async with httpx.AsyncClient() as client:
        if reset_prefix_cache:
            response = await client.post(f"{base_url}/reset_prefix_cache", timeout=60)
            _require(response.status_code == 200, f"prefix cache reset failed: HTTP {response.status_code}")
        for doc in list(warm_docs)[: cfg.warmup_requests]:
            await stream_request(client, base_url, condition, messages_for(condition, doc, full_skill), cfg)


# --------------------------------------------------------------------------- scoring and statistics


def score_output(condition: str, text: str, gold: Mapping[str, Any], validator: Any) -> dict[str, Any]:
    parsed = engine.parse_json_object(text)
    gold_receipt = engine.cord_to_expense(gold)
    if condition in ("base", "lora"):
        cord = engine.score_example(parsed, gold)
        expense = engine.score_expense({"receipts": [engine.cord_to_expense(parsed)]} if parsed is not None else None,
                                       gold_receipt, validator)
        schema_valid = parsed is not None  # CORD has no formal schema: a JSON object is schema-valid
    else:
        cord = None
        expense = engine.score_expense(parsed, gold_receipt, validator)
        schema_valid = expense["schema_valid"]
    return {"valid_json": parsed is not None, "schema_valid": schema_valid, "cord": cord, "expense": expense,
            "exact_match": expense_exact(expense)}


def expense_exact(score: Mapping[str, Any]) -> bool:
    """Whole receipt right: every gold item matched, nothing extra, every scalar right or rightly absent."""
    scalars_ok = all(v["correct"] if v["gold"] else not v["pred"] for v in score["scalars"].values())
    return bool(score["schema_valid"] and score["items_tp"] == score["items_gold"] == score["items_pred"] and scalars_ok)


def _prf(tp: float, pred: float, gold: float) -> tuple[float, float, float]:
    p = tp / pred if pred else 0.0
    r = tp / gold if gold else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)


def quality(scored: Sequence[Mapping[str, Any]]) -> dict[str, float | None]:
    expense = [s["expense"] for s in scored]
    tp = sum(e["items_tp"] for e in expense) + sum(v["correct"] for e in expense for v in e["scalars"].values())
    pred = sum(e["items_pred"] for e in expense) + sum(v["pred"] for e in expense for v in e["scalars"].values())
    gold = sum(e["items_gold"] for e in expense) + sum(v["gold"] for e in expense for v in e["scalars"].values())
    p, r, f1 = _prf(tp, pred, gold)
    totals = [e["scalars"]["total"] for e in expense if e["scalars"]["total"]["gold"]]
    metrics: dict[str, float | None] = {
        "main_f1": f1, "main_precision": p, "main_recall": r,
        "total_accuracy": sum(t["correct"] for t in totals) / len(totals) if totals else None,
        "exact_match": sum(s["exact_match"] for s in scored) / len(scored),
        "schema_valid": sum(s["schema_valid"] for s in scored) / len(scored),
        "valid_json": sum(s["valid_json"] for s in scored) / len(scored),
    }
    cord = [s["cord"] for s in scored if s["cord"] is not None]
    if len(cord) == len(scored):
        cp, cr, cf = _prf(sum(c["tp"] for c in cord), sum(c["pred_fields"] for c in cord), sum(c["gold_fields"] for c in cord))
        present = [c for c in cord if c["total_present"]]
        metrics.update({"cord_f1": cf, "cord_precision": cp, "cord_recall": cr,
                        "cord_total_exact": sum(c["total_exact"] for c in present) / len(present) if present else None,
                        "cord_document_exact": sum(c["document_exact"] for c in cord) / len(cord)})
    return metrics


def bootstrap(scored: Sequence[Mapping[str, Any]], samples: int, seed: int) -> dict[str, tuple[float, float]]:
    rng = random.Random(seed)
    draws: dict[str, list[float]] = {}
    for _ in range(samples):
        resampled = [scored[rng.randrange(len(scored))] for _ in scored]
        for key, value in quality(resampled).items():
            if value is not None:
                draws.setdefault(key, []).append(value)
    return {key: _interval(values) for key, values in draws.items()}


def paired_bootstrap(a: Sequence[Mapping[str, Any]], b: Sequence[Mapping[str, Any]], samples: int,
                     seed: int) -> dict[str, dict[str, float]]:
    _require(len(a) == len(b), "paired bootstrap needs the same documents in the same order")
    rng = random.Random(seed)
    point_a, point_b = quality(a), quality(b)
    draws: dict[str, list[float]] = {}
    for _ in range(samples):
        picks = [rng.randrange(len(a)) for _ in a]
        qa, qb = quality([a[i] for i in picks]), quality([b[i] for i in picks])
        for key in qa:
            if qa[key] is not None and qb.get(key) is not None:
                draws.setdefault(key, []).append(qa[key] - qb[key])
    return {key: {"delta": point_a[key] - point_b[key], "ci95": _interval(values)}
            for key, values in draws.items() if point_a.get(key) is not None and point_b.get(key) is not None}


def _interval(values: Sequence[float]) -> tuple[float, float]:
    ordered = sorted(values)
    return ordered[int(0.025 * (len(ordered) - 1))], ordered[int(math.ceil(0.975 * (len(ordered) - 1)))]


def _pct(values: Sequence[float], q: float) -> float | None:
    clean = sorted(v for v in values if v is not None)
    if not clean:
        return None
    position = (len(clean) - 1) * q
    low, high = math.floor(position), math.ceil(position)
    return clean[low] + (clean[high] - clean[low]) * (position - low)


def _stats(values: Sequence[float]) -> dict[str, float | None]:
    clean = [v for v in values if v is not None]
    return {"mean": statistics.mean(clean) if clean else None, "p50": _pct(clean, 0.5), "p95": _pct(clean, 0.95)}


# --------------------------------------------------------------------------- orchestration


def run(cfg: BenchConfig, docs: Sequence[Mapping[str, Any]], warm_docs: Sequence[Mapping[str, Any]],
        full_skill: str, output_schema: Mapping[str, Any], *, quick: bool, provenance: Mapping[str, Any],
        training: Mapping[str, Any]) -> dict[str, Any]:
    """Synchronous entry point; safe inside a notebook whose event loop is already running."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, _run(cfg, docs, warm_docs, full_skill, output_schema, quick=quick,
                                             provenance=provenance, training=training)).result()


async def _run(cfg: BenchConfig, docs: Sequence[Mapping[str, Any]], warm_docs: Sequence[Mapping[str, Any]],
               full_skill: str, output_schema: Mapping[str, Any], *, quick: bool, provenance: Mapping[str, Any],
               training: Mapping[str, Any]) -> dict[str, Any]:
    from jsonschema import Draft202012Validator

    cfg.validate()
    docs = list(docs)[: cfg.quick_docs] if quick else list(docs)
    repeats = 1 if quick else cfg.repeats
    out = Path(cfg.results_dir) / ("quick" if quick else "full")
    out.mkdir(parents=True, exist_ok=True)
    validator = Draft202012Validator(output_schema)
    rows: list[dict[str, Any]] = []
    runs: list[dict[str, Any]] = []
    servers_meta: dict[str, Any] = {}
    clock = time.perf_counter()

    for server, spec in SERVERS.items():
        with Server(cfg, server, out) as live:
            servers_meta[server] = {"start_seconds": live.start_seconds, "command": " ".join(server_command(cfg, server))}
            for condition in spec["conditions"]:
                for repeat in range(1, repeats + 1):
                    for concurrency in cfg.concurrencies:
                        await warm(cfg, live.base_url, condition, warm_docs, full_skill,
                                   reset_prefix_cache=spec["prefix_cache"])
                        result = await timed_run(cfg, live.base_url, condition, docs, full_skill, concurrency)
                        results = result["results"]
                        runs.append({"condition": condition, "server": server, "repeat": repeat,
                                     "concurrency": concurrency, "documents": len(results), "wall_s": result["wall_s"],
                                     "docs_per_s": len(results) / result["wall_s"],
                                     "output_tokens_per_s": sum(r["completion_tokens"] for r in results) / result["wall_s"],
                                     "kv_cache_usage_peak": result["kv_cache_usage_peak"],
                                     "prefix_cache_hit_rate": result["prefix_cache_hit_rate"],
                                     "gpu_memory_used_mib": result["gpu_memory_used_mib"]})
                        gold = {d["doc_id"]: d["gold"] for d in docs}
                        for r in results:
                            scored = score_output(condition, r["text"], gold[r["doc_id"]], validator)
                            rows.append({"condition": condition, "server": server, "repeat": repeat,
                                         "concurrency": concurrency, **r, **scored})
                        last = runs[-1]
                        print(f"[{condition}] repeat {repeat}/{repeats} | concurrency {concurrency} | "
                              f"{len(results)} docs in {result['wall_s']:.1f}s | {last['docs_per_s']:.2f} docs/s | "
                              f"TTFT p50 {_pct([r['ttft_s'] for r in results], 0.5):.3f}s | "
                              f"latency p50 {_pct([r['latency_s'] for r in results], 0.5):.2f}s", flush=True)
    elapsed = time.perf_counter() - clock
    report = summarize(cfg, rows, runs, docs, quick=quick, provenance=provenance, training=training,
                       servers=servers_meta, elapsed_s=elapsed)
    write_outputs(cfg, out, rows, runs, report)
    return report


def quality_rows(rows: Sequence[Mapping[str, Any]], condition: str) -> list[Mapping[str, Any]]:
    """Quality comes from the first repeat at concurrency 1, ordered by document."""
    selected = [r for r in rows if r["condition"] == condition and r["repeat"] == 1 and r["concurrency"] == 1]
    return sorted(selected, key=lambda r: r["doc_id"])


def summarize(cfg: BenchConfig, rows: Sequence[Mapping[str, Any]], runs: Sequence[Mapping[str, Any]],
              docs: Sequence[Mapping[str, Any]], *, quick: bool, provenance: Mapping[str, Any],
              training: Mapping[str, Any], servers: Mapping[str, Any], elapsed_s: float) -> dict[str, Any]:
    # skill_cached and skill_uncached must generate the same text; quality is then scored on both but they agree.
    cached, uncached = quality_rows(rows, "skill_cached"), quality_rows(rows, "skill_uncached")
    identical = [a["text"] == b["text"] for a, b in zip(cached, uncached)]
    sample = identical[: cfg.identical_output_sample]
    _require(all(sample), f"skill_cached and skill_uncached outputs differ on the first {len(sample)} documents: "
                          f"{[c['doc_id'] for c, same in zip(cached, sample) if not same]}")

    conditions: dict[str, Any] = {}
    for condition in CONDITIONS:
        scored = quality_rows(rows, condition)
        speed = [r for r in rows if r["condition"] == condition and r["concurrency"] == 1]
        per_repeat = {}
        for repeat in sorted({r["repeat"] for r in speed}):
            subset = [r for r in speed if r["repeat"] == repeat]
            per_repeat[repeat] = {"ttft_p50": _pct([r["ttft_s"] for r in subset], 0.5),
                                  "latency_p50": _pct([r["latency_s"] for r in subset], 0.5)}
        throughput = {}
        for concurrency in cfg.concurrencies:
            matching = [r for r in runs if r["condition"] == condition and r["concurrency"] == concurrency]
            throughput[concurrency] = {
                "docs_per_s": statistics.mean(r["docs_per_s"] for r in matching),
                "docs_per_s_range": (min(r["docs_per_s"] for r in matching), max(r["docs_per_s"] for r in matching)),
                "output_tokens_per_s": statistics.mean(r["output_tokens_per_s"] for r in matching),
                "kv_cache_usage_peak": max((r["kv_cache_usage_peak"] for r in matching if r["kv_cache_usage_peak"] is not None), default=None),
                "gpu_memory_used_mib": max((r["gpu_memory_used_mib"] for r in matching if r["gpu_memory_used_mib"] is not None), default=None),
                "prefix_cache_hit_rate": statistics.mean([r["prefix_cache_hit_rate"] for r in matching if r["prefix_cache_hit_rate"] is not None]) if any(r["prefix_cache_hit_rate"] is not None for r in matching) else None,
            }
        top = max(cfg.concurrencies)
        cost = {f"usd_per_1000_docs_at_{c}": 1000 / throughput[c]["docs_per_s"] / 3600 * cfg.gpu_hour_usd
                for c in cfg.concurrencies}
        conditions[condition] = {
            "quality": quality(scored),
            "quality_ci95": bootstrap(scored, cfg.bootstrap_samples, cfg.seed),
            "prompt_tokens_mean": statistics.mean(r["prompt_tokens"] for r in scored),
            "output_tokens_mean": statistics.mean(r["completion_tokens"] for r in scored),
            "ttft_s": _stats([r["ttft_s"] for r in speed]),
            "latency_s": _stats([r["latency_s"] for r in speed]),
            "decode_tokens_per_s": _stats([r["decode_tokens_per_s"] for r in speed]),
            "speed_by_repeat": per_repeat,
            "throughput": throughput,
            "cost": cost,
            "top_concurrency": top,
        }

    lora, cached_q = quality_rows(rows, "lora"), quality_rows(rows, "skill_cached")
    top = max(cfg.concurrencies)
    saving = conditions["skill_cached"]["cost"][f"usd_per_1000_docs_at_{top}"] / 1000 - \
        conditions["lora"]["cost"][f"usd_per_1000_docs_at_{top}"] / 1000
    training_cost = training.get("train_seconds", 0) / 3600 * cfg.gpu_hour_usd
    tuning_seconds = training.get("train_seconds_including_tuning")
    with_tuning = tuning_seconds / 3600 * cfg.gpu_hour_usd if tuning_seconds is not None else None
    return {
        "mode": "quick" if quick else "full", "documents": len(docs), "repeats": max(r["repeat"] for r in rows),
        "provenance": dict(provenance), "config": {k: v for k, v in asdict(cfg).items() if k != "server_command_override"},
        "servers": dict(servers), "elapsed_s": elapsed_s,
        "skill_outputs_identical": {"sample_checked": len(sample), "identical_in_sample": sum(sample),
                                    "identical_all": sum(identical), "documents": len(identical)},
        "conditions": conditions,
        "lora_vs_skill_cached": paired_bootstrap(lora, cached_q, cfg.bootstrap_samples, cfg.seed),
        "one_time_lora_cost": {
            "train_seconds": training.get("train_seconds"),
            "train_seconds_including_tuning": training.get("train_seconds_including_tuning"),
            "usd": training_cost, "usd_including_tuning": with_tuning,
            "saving_per_doc_vs_skill_cached_usd": saving,
            "break_even_docs": math.ceil(training_cost / saving) if saving > 0 else None,
            "break_even_docs_including_tuning": math.ceil(with_tuning / saving) if saving > 0 and with_tuning is not None else None,
        },
    }


def estimate_full_minutes(report: Mapping[str, Any], cfg: BenchConfig, documents: int) -> dict[str, float]:
    """Full-run GPU time extrapolated from a quick run's measured latency and throughput."""
    total = 0.0
    for name, c in report["conditions"].items():
        c1 = c["latency_s"]["mean"] or 0.0
        per_repeat = cfg.warmup_requests * c1 * len(cfg.concurrencies)
        for concurrency in cfg.concurrencies:
            per_repeat += documents / c["throughput"][concurrency]["docs_per_s"]
        total += per_repeat * cfg.repeats
    total += sum(s["start_seconds"] for s in report["servers"].values())
    return {"minutes": total / 60, "usd": total / 3600 * cfg.gpu_hour_usd}


# --------------------------------------------------------------------------- outputs


def _fmt(value: Any, digits: int = 3) -> str:
    if value is None:
        return "n/a"
    if isinstance(value, float):
        return f"{value:.{digits}f}"
    return str(value)


def _ci(point: float | None, interval: tuple[float, float] | None) -> str:
    if point is None:
        return "n/a"
    return f"{point:.3f} [{interval[0]:.3f}, {interval[1]:.3f}]" if interval else f"{point:.3f}"


def headline(report: Mapping[str, Any]) -> list[str]:
    top = max(report["config"]["concurrencies"])
    lines = [f"| Condition | Main accuracy (field F1, 95% CI) | OOD accuracy | Prompt tokens | TTFT p50 (s) | Latency p50 (s) | Throughput @{top} (docs/s) |",
             "|---|---|---|---|---|---|---|"]
    for name, c in report["conditions"].items():
        lines.append(f"| {name} | {_ci(c['quality']['main_f1'], c['quality_ci95'].get('main_f1'))} | n/a (no OOD set) | "
                     f"{c['prompt_tokens_mean']:.0f} | {_fmt(c['ttft_s']['p50'])} | {_fmt(c['latency_s']['p50'], 2)} | "
                     f"{c['throughput'][top]['docs_per_s']:.2f} |")
    return lines


def plain_read(report: Mapping[str, Any]) -> list[str]:
    """Generated from the numbers; no judgement is written that the data does not support."""
    c = report["conditions"]
    top = max(report["config"]["concurrencies"])
    diff = report["lora_vs_skill_cached"].get("main_f1")
    lines = []
    top_f1 = max(v["quality"]["main_f1"] for v in c.values())
    leaders = [k for k, v in c.items() if abs(v["quality"]["main_f1"] - top_f1) < 1e-9]
    lines.append(f"- Highest main accuracy: **{' / '.join(leaders)}** ({top_f1:.3f})"
                 + (" (tied)." if len(leaders) > 1 else "."))
    if diff:
        lo, hi = diff["ci95"]
        verdict = ("LoRA is more accurate" if lo > 0 else "the skill is more accurate" if hi < 0
                   else "the difference is not statistically clear")
        lines.append(f"- LoRA vs cached skill, field F1: {diff['delta']:+.3f} (95% CI {lo:+.3f} to {hi:+.3f}); {verdict}.")
    lp, sp = c["lora"]["prompt_tokens_mean"], c["skill_cached"]["prompt_tokens_mean"]
    lines.append(f"- Prompt tokens: LoRA {lp:.0f} vs skill {sp:.0f} ({(lp - sp) / sp:+.0%}).")
    for key, label in (("ttft_s", "TTFT p50"), ("latency_s", "Latency p50")):
        lines.append(f"- {label}: " + ", ".join(f"{k} {_fmt(v[key]['p50'])}s" for k, v in c.items()) + ".")
    lines.append(f"- Throughput at concurrency {top}: " + ", ".join(f"{k} {v['throughput'][top]['docs_per_s']:.2f} docs/s" for k, v in c.items()) + ".")
    lines.append(f"- Cost per 1,000 documents at concurrency {top}: " + ", ".join(
        f"{k} ${v['cost'][f'usd_per_1000_docs_at_{top}']:.3f}" for k, v in c.items()) + ".")
    be = report["one_time_lora_cost"]
    if be["break_even_docs"] is None:
        lines.append("- Break-even: none. LoRA is not cheaper per document than the cached skill at this concurrency.")
    else:
        tuning = (f"{be['break_even_docs_including_tuning']:,} including tuning (${be['usd_including_tuning']:.2f})"
                  if be["break_even_docs_including_tuning"] is not None else "tuning logs unavailable, so no figure including tuning")
        lines.append(f"- Break-even vs cached skill: {be['break_even_docs']:,} documents for the final training run "
                     f"(${be['usd']:.2f}); {tuning}.")
    return lines


def write_outputs(cfg: BenchConfig, out: Path, rows: Sequence[Mapping[str, Any]], runs: Sequence[Mapping[str, Any]],
                  report: Mapping[str, Any]) -> None:
    fields = ["condition", "server", "repeat", "concurrency", "doc_id", "prompt_tokens", "completion_tokens", "ttft_s",
              "latency_s", "decode_tokens_per_s", "valid_json", "schema_valid", "exact_match",
              "fields_correct", "fields_predicted", "fields_gold"]
    with (out / "benchmark_runs.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for r in rows:
            e = r["expense"]
            writer.writerow({**{k: r.get(k) for k in fields[:12]}, "exact_match": r["exact_match"],
                             "fields_correct": e["items_tp"] + sum(v["correct"] for v in e["scalars"].values()),
                             "fields_predicted": e["items_pred"] + sum(v["pred"] for v in e["scalars"].values()),
                             "fields_gold": e["items_gold"] + sum(v["gold"] for v in e["scalars"].values())})
    summary_fields = ["condition"]
    summary_rows = []
    for name, c in report["conditions"].items():
        row = {"condition": name, "prompt_tokens_mean": c["prompt_tokens_mean"], "output_tokens_mean": c["output_tokens_mean"]}
        for key, value in c["quality"].items():
            row[key] = value
            interval = c["quality_ci95"].get(key)
            row[f"{key}_ci95_low"], row[f"{key}_ci95_high"] = interval if interval else (None, None)
        for metric in ("ttft_s", "latency_s", "decode_tokens_per_s"):
            for stat, value in c[metric].items():
                row[f"{metric}_{stat}"] = value
        for concurrency, t in c["throughput"].items():
            row[f"docs_per_s_c{concurrency}"] = t["docs_per_s"]
            row[f"docs_per_s_c{concurrency}_min"], row[f"docs_per_s_c{concurrency}_max"] = t["docs_per_s_range"]
            row[f"output_tokens_per_s_c{concurrency}"] = t["output_tokens_per_s"]
            row[f"kv_cache_usage_peak_c{concurrency}"] = t["kv_cache_usage_peak"]
            row[f"gpu_memory_used_mib_c{concurrency}"] = t["gpu_memory_used_mib"]
        row.update(c["cost"])
        summary_rows.append(row)
        summary_fields += [k for k in row if k not in summary_fields]
    with (out / "benchmark_summary.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=summary_fields)
        writer.writeheader()
        writer.writerows(summary_rows)
    (out / "benchmark_runs_per_run.json").write_text(json.dumps(list(runs), indent=2))
    (out / "benchmark_report.json").write_text(json.dumps(report, indent=2, default=str))
    charts(report, out)
    (out / "DECISIONS.md").write_text(decisions(report), encoding="utf-8")
    (out / "benchmark_report.md").write_text(markdown(report), encoding="utf-8")


def charts(report: Mapping[str, Any], out: Path) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    names = list(report["conditions"])
    c = report["conditions"]
    figure, axis = plt.subplots(figsize=(7, 4))
    points = [c[n]["quality"]["main_f1"] for n in names]
    intervals = [c[n]["quality_ci95"].get("main_f1", (p, p)) for n, p in zip(names, points)]
    axis.bar(names, points, yerr=[[p - i[0] for p, i in zip(points, intervals)], [i[1] - p for p, i in zip(points, intervals)]], capsize=6)
    axis.set_ylabel("Field F1 (95% CI)"); axis.set_ylim(0, 1); axis.set_title("Accuracy by condition")
    figure.tight_layout(); figure.savefig(out / "accuracy_by_condition.png", dpi=150); plt.close(figure)

    figure, axis = plt.subplots(figsize=(7, 4))
    axis.bar(names, [c[n]["prompt_tokens_mean"] for n in names])
    axis.set_ylabel("Mean prompt tokens"); axis.set_title("Prompt tokens by condition")
    figure.tight_layout(); figure.savefig(out / "prompt_tokens_by_condition.png", dpi=150); plt.close(figure)

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    for axis, metric, label in ((axes[0], "ttft_s", "TTFT (s)"), (axes[1], "latency_s", "Latency (s)")):
        p50 = [c[n][metric]["p50"] or 0 for n in names]
        p95 = [c[n][metric]["p95"] or 0 for n in names]
        positions = range(len(names))
        axis.bar([x - 0.2 for x in positions], p50, width=0.4, label="p50")
        axis.bar([x + 0.2 for x in positions], p95, width=0.4, label="p95")
        axis.set_xticks(list(positions)); axis.set_xticklabels(names); axis.set_ylabel(label); axis.legend()
    axes[0].set_title("Time to first token (concurrency 1)"); axes[1].set_title("End-to-end latency (concurrency 1)")
    figure.tight_layout(); figure.savefig(out / "ttft_and_latency.png", dpi=150); plt.close(figure)

    figure, axis = plt.subplots(figsize=(7, 4))
    for n in names:
        levels = sorted(c[n]["throughput"], key=int)
        axis.plot([int(x) for x in levels], [c[n]["throughput"][x]["docs_per_s"] for x in levels], marker="o", label=n)
    axis.set_xlabel("Concurrency"); axis.set_ylabel("Documents per second"); axis.set_xscale("log", base=2)
    axis.set_title("Throughput vs concurrency"); axis.legend()
    figure.tight_layout(); figure.savefig(out / "throughput_vs_concurrency.png", dpi=150); plt.close(figure)


def markdown(report: Mapping[str, Any]) -> str:
    c = report["conditions"]
    lines = [f"# Four-way benchmark: receipts (CORD) · {report['mode']} run", "", *headline(report), "",
             "## What won, and where", *plain_read(report), "", "## Quality detail (95% bootstrap CIs)", "",
             "| Condition | Field P | Field R | Total accuracy | Exact match | Schema valid | CORD F1 |", "|---|---|---|---|---|---|---|"]
    for name, x in c.items():
        q, ci = x["quality"], x["quality_ci95"]
        lines.append(f"| {name} | {_ci(q['main_precision'], ci.get('main_precision'))} | {_ci(q['main_recall'], ci.get('main_recall'))} | "
                     f"{_ci(q['total_accuracy'], ci.get('total_accuracy'))} | {_ci(q['exact_match'], ci.get('exact_match'))} | "
                     f"{_ci(q['schema_valid'], ci.get('schema_valid'))} | {_ci(q.get('cord_f1'), ci.get('cord_f1'))} |")
    lines += ["", "## Speed at concurrency 1 (all repeats)", "", "| Condition | TTFT mean / p50 / p95 | Latency mean / p50 / p95 | Decode tok/s p50 | p50 latency across repeats |", "|---|---|---|---|---|"]
    for name, x in c.items():
        spread = [v["latency_p50"] for v in x["speed_by_repeat"].values()]
        lines.append(f"| {name} | {_fmt(x['ttft_s']['mean'])} / {_fmt(x['ttft_s']['p50'])} / {_fmt(x['ttft_s']['p95'])} | "
                     f"{_fmt(x['latency_s']['mean'], 2)} / {_fmt(x['latency_s']['p50'], 2)} / {_fmt(x['latency_s']['p95'], 2)} | "
                     f"{_fmt(x['decode_tokens_per_s']['p50'], 1)} | {min(spread):.2f}–{max(spread):.2f} s |")
    levels = report["config"]["concurrencies"]
    lines += ["", "## Scale", "", "| Condition | " + " | ".join(f"docs/s @{l} (min–max)" for l in levels) + " | Peak KV cache use | GPU memory (MiB) |",
              "|---|" + "---|" * (len(levels) + 2)]
    for name, x in c.items():
        t = x["throughput"]
        kv = max((t[l]["kv_cache_usage_peak"] for l in levels if t[l]["kv_cache_usage_peak"] is not None), default=None)
        mem = max((t[l]["gpu_memory_used_mib"] for l in levels if t[l]["gpu_memory_used_mib"] is not None), default=None)
        lines.append(f"| {name} | " + " | ".join(f"{t[l]['docs_per_s']:.2f} ({t[l]['docs_per_s_range'][0]:.2f}–{t[l]['docs_per_s_range'][1]:.2f})" for l in levels)
                     + f" | {_fmt(kv)} | {_fmt(mem)} |")
    be = report["one_time_lora_cost"]
    same = report["skill_outputs_identical"]
    lines += ["", "## Cost and break-even", "",
              f"- GPU price: ${report['config']['gpu_hour_usd']}/GPU-hour ({report['config']['gpu_hour_note']})",
              f"- Final LoRA training: {_fmt(be['train_seconds'], 0)} s (${be['usd']:.2f}); including tuning: "
              + (f"{_fmt(be['train_seconds_including_tuning'], 0)} s (${be['usd_including_tuning']:.2f})."
                 if be["usd_including_tuning"] is not None else "not available (tuning logs were lost with an earlier runtime)."),
              f"- Saving per 1,000 documents vs cached skill: ${1000 * be['saving_per_doc_vs_skill_cached_usd']:.4f}.",
              "", "## Caveats", "",
              f"- skill_cached and skill_uncached produced identical text on {same['identical_in_sample']}/{same['sample_checked']} sampled "
              f"documents ({same['identical_all']}/{same['documents']} overall).",
              "- No OOD set exists for the CORD receipt skill; the OOD column is n/a.",
              "- All four conditions run BF16 through vLLM; the adapter was trained on the 4-bit (NF4) base.",
              f"- LoRA served as: {report['config']['lora_mode']}.",
              "- See DECISIONS.md for every choice made where the spec was ambiguous.",
              "", "## Provenance", "", "```json", json.dumps(report["provenance"], indent=2), "```",
              "", "Charts: accuracy_by_condition.png, prompt_tokens_by_condition.png, ttft_and_latency.png, throughput_vs_concurrency.png"]
    return "\n".join(lines) + "\n"


def decisions(report: Mapping[str, Any]) -> str:
    cfg = report["config"]
    return "\n".join([
        "# DECISIONS",
        "",
        "Choices made where the benchmark spec was ambiguous, with the simplest option taken.",
        "",
        f"1. **Precision.** All four conditions run `{cfg['model_id']}` in {cfg['dtype']} through vLLM. The adapter was trained "
        "with QLoRA on the 4-bit NF4 base; serving it on BF16 weights is standard QLoRA deployment but not bit-identical to training.",
        f"2. **LoRA serving.** `{cfg['lora_mode']}`: 'adapter' uses vLLM `--enable-lora`; 'merged' serves the adapter merged into BF16 weights.",
        "3. **Main accuracy.** The skill conditions use the repo's receipt-to-expense skill unchanged, which answers in an expense-report "
        "format. The only metric defined for all four conditions is the skill-format field F1 (line items + subtotal, tax, service "
        "charge, total, per the skill's evaluation_plan.json); CORD-format answers are converted with the same code as the gold labels. "
        "The baseline CORD scorer is also reported for base and lora.",
        "4. **Schema validity.** Skill outputs: validates against the skill's output_schema.json. CORD outputs have no formal schema, "
        "so schema-valid means the output parses as a JSON object.",
        "5. **Exact match.** Every gold line item matched with nothing extra, and every scalar correct or correctly absent.",
        "6. **Test set.** The pinned CORD test parquet; its sha256 is asserted before running and recorded under provenance. "
        "No OOD set exists for this skill (SROIE is excluded by project decision).",
        "7. **Thinking mode.** Qwen3-VL-8B-Instruct has no thinking mode; `enable_thinking=False` is sent anyway.",
        "8. **Warmup and caching fairness.** Warmup uses validation receipts, never test receipts. Before every timed run the cached "
        "server's prefix cache is reset and re-warmed, so only the shared skill prefix is cached. vLLM's multimodal processor cache is "
        "disabled on every server so repeated documents do not get cheaper.",
        "9. **Quality source.** Quality is scored on the first repeat at concurrency 1. skill_cached and skill_uncached are asserted "
        f"identical on the first {report['skill_outputs_identical']['sample_checked']} documents.",
        f"10. **Cost.** ${cfg['gpu_hour_usd']}/GPU-hour ({cfg['gpu_hour_note']}). Cost per document = GPU time per document at the "
        "stated concurrency (1 / throughput). Training cost uses the logged wall-clock seconds at the same rate.",
        "11. **Image preprocessing.** Receipts are resized to the same 1280px long side used for training and evaluation and sent as PNG; "
        "vLLM's Qwen3-VL processor applies its defaults after that.",
        f"12. **Serving stack.** vLLM {report['provenance'].get('vllm', 'n/a')} (torch {report['provenance'].get('vllm_torch', 'n/a')}) in its own "
        "virtual environment, separate from the Unsloth training stack; one server configuration per row of the SERVERS table, "
        "identical except for prefix caching and the adapter.",
        "",
    ])

## Cell 20: four-way benchmark (final)
**base** (short prompt) vs **skill_uncached** vs **skill_cached** (vLLM prefix caching, warmed) vs **lora** (short prompt + adapter),
all BF16 through vLLM on the locked CORD test split (sha256 asserted). Quality with 95% bootstrap CIs, cost per 1,000 documents,
TTFT / latency / decode speed at concurrency 1, throughput at 1, 8 and 32, and LoRA break-even. Runs `QUICK = True` first
(5 receipts per condition) and prints the full-run GPU estimate; set `QUICK = False` only after you approve it.
Writes `results/{quick,full}/benchmark_runs.csv`, `benchmark_summary.csv`, `benchmark_report.md`, `DECISIONS.md` and four charts.

In [ ]:
import hashlib, importlib, json, subprocess, sys, time
from pathlib import Path
sys.path.insert(0, "/content")
import dapta_vlm_engine as engine, dapta_bench as bench
importlib.reload(engine); importlib.reload(bench)
from datasets import Image as ImageFeature, load_dataset
from huggingface_hub import hf_hub_download

QUICK = True            # 5 test receipts per condition; set False only after approving the full-run estimate
LORA_MODE = "adapter"   # "adapter" = vLLM --enable-lora; "merged" = run cell 18 with MERGE_ADAPTER = True first
GPU_HOUR_USD = 1.50     # price of one A100 40GB GPU-hour; used for cost and break-even
ROOT = Path("/content/dapta_lora")
TEST_FILE, TEST_SHA256 = "data/test-00000-of-00001-9c204eb3f4e11791.parquet", "51c65f1788faff392abe2a0b55b023eb23e9be551c509138eaa3a832514224e7"
VALIDATION_FILE = "data/validation-00000-of-00001-cc3c5779fe22e8ca.parquet"

used = bench.gpu_memory_used_mib()
assert used is not None and used < 2500, (f"{used} MiB of GPU memory is in use. Restart the kernel (files in /content are kept), "
                                         "then run cells 17-20 only.")
CFG_E = engine.EngineConfig().validate()

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

test_path = hf_hub_download(CFG_E.dataset_id, TEST_FILE, repo_type="dataset", revision=CFG_E.dataset_revision)
assert sha256_file(test_path) == TEST_SHA256, "the CORD test split does not match the locked hash"
test = load_dataset("parquet", data_files=test_path, split="train").cast_column("image", ImageFeature())
assert len(test) == 100
val_path = hf_hub_download(CFG_E.dataset_id, VALIDATION_FILE, repo_type="dataset", revision=CFG_E.dataset_revision)
validation = load_dataset("parquet", data_files=val_path, split="train").cast_column("image", ImageFeature())
DOCS = bench.encode_docs(test, range(len(test)), CFG_E)
WARM = bench.encode_docs(validation, [97, 98, 99], CFG_E)  # warmup never touches test receipts

SKILL_SPEC = json.loads(r'''{
  "name": "receipt_to_expense_report",
  "version": "1.0.0",
  "description": "Convert one or more receipt images into a normalized expense report containing only visible evidence.",
  "input": {
    "modality": "receipt_images",
    "cardinality": "one_or_more",
    "constraints": [
      "each image is processed as one independent receipt object",
      "no external knowledge, lookup, or context beyond the image pixels"
    ]
  },
  "output": "strict JSON validating against output_schema; receipts array length equals number of input images",
  "extraction_rules": [
    "Extract only text visibly printed on the receipt; never guess, impute, or complete partial values.",
    "Use null for any field not explicitly visible; never use empty strings, placeholders, or defaults.",
    "merchant: merchant/store name exactly as visible.",
    "address: address text transcribed as printed; null if no address is visible.",
    "transaction_date: visible transaction date normalized to ISO 8601 YYYY-MM-DD; null if absent or unresolvably ambiguous.",
    "currency: ISO 4217 alpha-3 code only when a currency code or unambiguous symbol is explicitly printed; otherwise null; never infer from locale, language, merchant, or address.",
    "subtotal, tax, service_charge, total: numeric values as printed; null when the corresponding labeled line is absent.",
    "line_items: one entry per visible purchased item; description as printed; quantity as printed number else null; amount is the printed line total."
  ],
  "normalization": {
    "amounts": "plain decimal numbers; strip currency symbols and thousands separators",
    "dates": "YYYY-MM-DD",
    "strings": "trim leading/trailing whitespace; collapse internal whitespace runs"
  },
  "exclusions": [
    "no policy approval determination",
    "no reimbursement eligibility determination",
    "no expense category or GL code assignment",
    "no currency inference",
    "no fields beyond output_schema",
    "no commentary, markdown, or chain-of-thought in output"
  ],
  "training_data": {
    "datasets": [
      "CORD v2 (pinned)",
      "SROIE (pinned)"
    ],
    "label_mapping": {
      "CORD_v2": {
        "menu": "line_items",
        "service": "service_charge",
        "sub_total": "subtotal",
        "tax": "tax",
        "total": "total"
      },
      "SROIE": {
        "address": "address",
        "company": "merchant",
        "date": "transaction_date",
        "total": "total"
      }
    }
  }
}''')
OUTPUT_SCHEMA = json.loads(r'''{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "receipts": {
      "type": "array",
      "minItems": 1,
      "items": {
        "type": "object",
        "additionalProperties": false,
        "properties": {
          "merchant": {
            "type": ["string", "null"]
          },
          "address": {
            "type": ["string", "null"]
          },
          "transaction_date": {
            "type": ["string", "null"],
            "pattern": "^\\d{4}-\\d{2}-\\d{2}$"
          },
          "currency": {
            "type": ["string", "null"],
            "pattern": "^[A-Z]{3}$"
          },
          "subtotal": {
            "type": ["number", "null"]
          },
          "tax": {
            "type": ["number", "null"]
          },
          "service_charge": {
            "type": ["number", "null"]
          },
          "total": {
            "type": ["number", "null"]
          },
          "line_items": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": false,
              "properties": {
                "description": {
                  "type": ["string", "null"]
                },
                "quantity": {
                  "type": ["number", "null"]
                },
                "amount": {
                  "type": ["number", "null"]
                }
              },
              "required": ["description", "quantity", "amount"]
            }
          }
        },
        "required": [
          "merchant",
          "address",
          "transaction_date",
          "currency",
          "subtotal",
          "tax",
          "service_charge",
          "total",
          "line_items"
        ]
      }
    }
  },
  "required": ["receipts"]
}''')
REVIEWED_SHA256 = {"skill_spec.json": "53e2752024304c966926f386974a6207d441f21b6de9aec2ee28dd2bf90ee89b", "output_schema.json": "8fbecf6888ae3320d5a17e548f9c63aeb7ca1c7d0af049dc6f1fee43c43d3823", "evaluation_plan.json": "81802ef4df48f953ddcbf8d946a9b88f8857bb049cd03e1b237dc2c4279dcc13"}
for name, value in (("skill_spec.json", SKILL_SPEC), ("output_schema.json", OUTPUT_SCHEMA)):
    assert hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest() == REVIEWED_SHA256[name]
FULL_SKILL = engine.full_skill_prompt(SKILL_SPEC, OUTPUT_SCHEMA)

full_summary = ROOT / "full" / "train_summary.json"
assert full_summary.exists(), "full training summary missing: run cell 13 first"
train_seconds = json.loads(full_summary.read_text())["train_seconds"]
tuning_files = [p for p in ROOT.glob("*/train_summary.json") if p.parent.name not in ("full", "tests")]
tuning_seconds = sum(json.loads(p.read_text())["train_seconds"] for p in tuning_files) if tuning_files else None
if LORA_MODE == "merged":
    assert (ROOT / "full" / "merged_bf16" / "config.json").exists(), "run cell 18 with MERGE_ADAPTER = True first"
versions = subprocess.run(["/content/vllm-env/bin/python", "-c", "import vllm, torch; print(vllm.__version__, torch.__version__)"],
                          capture_output=True, text=True, check=True).stdout.split()
PROVENANCE = {"test_file": TEST_FILE, "test_sha256": TEST_SHA256, "dataset_revision": CFG_E.dataset_revision,
              "test_documents": len(DOCS), "warmup_documents": "validation 97-99", "vllm": versions[0], "vllm_torch": versions[1],
              "adapter_config_sha256": sha256_file(ROOT / "full" / "adapter" / "adapter_config.json"),
              "engine_sha": "f5b60903a68c", "bench_sha": "7e7157ac15f9", "run_at": time.strftime("%Y-%m-%d %H:%M:%S %Z")}

CFG_B = bench.BenchConfig(lora_mode=LORA_MODE, gpu_hour_usd=GPU_HOUR_USD, results_dir=str(ROOT / "results")).validate()
REPORT = bench.run(CFG_B, DOCS, WARM, FULL_SKILL, OUTPUT_SCHEMA, quick=QUICK, provenance=PROVENANCE,
                   training={"train_seconds": train_seconds,
                             "train_seconds_including_tuning": train_seconds + tuning_seconds if tuning_seconds is not None else None})
out = Path(CFG_B.results_dir) / ("quick" if QUICK else "full")
print("\nFINAL BENCHMARK", "(QUICK RUN, 5 test receipts)" if QUICK else "(FULL RUN, 100 test receipts)")
for line in bench.headline(REPORT):
    print("FINAL ROW", line)
for line in bench.plain_read(REPORT):
    print("FINAL READ", line)
if QUICK:
    estimate = bench.estimate_full_minutes(REPORT, CFG_B, documents=len(DOCS))
    print(f"FINAL ESTIMATE full run: about {estimate['minutes']:.0f} minutes of A100 time (~${estimate['usd']:.2f}). "
          "Approve, then set QUICK = False and rerun this cell.")
print("Outputs:", sorted(p.name for p in out.iterdir()))
if "persist" in globals():
    persist([f"results/{out.name}/**"], f"four-way benchmark ({out.name})")
else:
    print("Kernel was restarted: results are only in", out, "- run cell 16 or copy them before the runtime ends.")
from IPython.display import Image as Show, display
for chart in ("accuracy_by_condition.png", "prompt_tokens_by_condition.png", "ttft_and_latency.png", "throughput_vs_concurrency.png"):
    display(Show(filename=str(out / chart)))